In [ ]:
# Shared repository paths (no model or dataset is loaded here).
from pathlib import Path as _RepositoryPath
import os as _repository_os
import sys as _repository_sys

_repository_hint = _repository_os.environ.get("TESI_REPOSITORY_ROOT")
_repository_starts = (
    [_RepositoryPath(_repository_hint).expanduser().resolve()]
    if _repository_hint else
    ([_RepositoryPath(__file__).resolve().parent] if "__file__" in globals() else [])
    + [_RepositoryPath.cwd().resolve()]
)
_repository_root = next(
    (candidate for start in _repository_starts for candidate in (start, *start.parents)
     if (candidate / "project_paths.py").is_file()), None
)
if _repository_root is None:
    raise FileNotFoundError(
        "Start Jupyter inside the repository or set TESI_REPOSITORY_ROOT to its root."
    )
if str(_repository_root) not in _repository_sys.path:
    _repository_sys.path.insert(0, str(_repository_root))
from project_paths import PROJECT_ROOT as TESI_PROJECT_ROOT
from project_paths import BASELINES_PATH as TESI_BASELINES_PATH


# RoutedModel — merged configurabile soltanto al primo layer

Dodici posizioni di routing. `MERGED_MODE` ammette tre valori:
- `off`: il layer encoder merged non puo' essere scelto in nessuna posizione.
- `auto`: merged disponibile soltanto al primo layer, con scelta libera del router.
- `impose`: primo layer obbligatoriamente merged; dal secondo solo SST, MNLI o skip.

Configurazione di questa prova: **MERGED_MODE = "impose"**.
Il backbone merged conserva soltanto il layer encoder 1, gli embedding e il pooler
mediati. Le modalita' riguardano le scelte encoder; embedding e pooler condivisi
mantengono la struttura attuale. I rami vietati non vengono eseguiti nel training.

Pesi concordati per questa prova:
- Classificazione: **1.15 comune a SST e MNLI**, senza separazione per task.
- Costo SST: **0.035** costante.
- Costo MNLI: **0.030 -> 0.050**, massimo all'epoca 5 su 8.
- Bonus entropico comune: **0.005 -> 0**, lungo tutte le otto epoche.

Formula della loss, progressioni, temperatura, learning-rate scheduler, dati,
percorsi e selezione dei due checkpoint mantengono la configurazione esistente.
Otto epoche, DEBUG_FRACTION = 1. Temperatura da 1 a 0.1 sull'intero budget.
Il checkpoint efficiente, se ammissibile su entrambi i task, viene valutato;
in sua assenza si usa il migliore per F1.

Modelli e risultati: RoutedModel/Models/<timestamp>/<timestamp>_seed_N e la stessa struttura in Results.
La modalita' merged e le scelte disponibili sono registrate nei risultati.
Le tracce JSONL conservano le quattro colonne e usano null per ogni logit vietato,
anche al primo layer. Nessuna esecuzione locale.


## Tre seed — esecuzione completa

Le celle seguenti definiscono le fasi; l'ultima cella avvia in sequenza i tre training
da zero e stampa/salva la tabella comune. Eseguire tutte le celle in ordine, anche
con nbconvert. I pesi della loss e MERGED_MODE restano quelli riportati sopra.
Gli esperti sono fissi; bootstrap sul test, separato dalla selezione su validation.
MNLI matched/mismatched esclusi. Dataset invariato.

Output: Models/<timestamp>/<timestamp>_seed_1 (2, 3) e la stessa struttura
in Results. Nella cartella Results/<timestamp>: expert_sst_predictions.csv,
expert_mnli_predictions.csv e statistical_summary.csv.

## Salvataggio e ripresa automatica
All’avvio viene controllata l’ultima cartella timestamp in `Results`. Se il suo `execution_manifest.json` indica un’esecuzione conclusa, ne viene creata una nuova; altrimenti viene riutilizzata la stessa cartella e vengono saltati i seed completati.
A fine epoca `Models/<timestamp>/<timestamp>_seed_N/training_resume.pt` viene sostituito atomicamente: contiene modello, optimizer, scheduler, stati casuali, storico, contatori e migliori pesi selezionati sulla validation. L’epoca interrotta viene rifatta dall’inizio. Se il training è completo, si ripete solo la valutazione rimasta incompleta. Dopo un riavvio del server occorre rilanciare il normale comando in coda.


In [ ]:
# Three independent routed trainings on the same frozen experts.
RUN_SEEDS = (42, 43, 44)
ROUTED_BASELINES_DIRECTORY = str(TESI_PROJECT_ROOT / 'Secondo_Esperimento/Baselines')
DATA_SEED = 42  # Dataset/subset identity does not change with the training seed.
BOOTSTRAP_RESAMPLES = 10000
BOOTSTRAP_SEED = 2026
LATENCY_WARMUP_BATCHES = 10
LATENCY_REPETITIONS = 3


In [ ]:
def input_fingerprint(input_ids, attention_mask, token_type_ids):
    """Same unpadded token identity used by the existing routed JSONL exports."""
    import hashlib
    import json
    tokens = [(int(i), int(t)) for i, t, m in
              zip(input_ids, token_type_ids, attention_mask) if m]
    return hashlib.sha256(json.dumps(tokens, separators=(",", ":")).encode("utf-8")).hexdigest()


def benchmark_inference(forward_batch, dataloader, device, *, warmup_batches=10, repetitions=3):
    """Time forward only, with CPU dispatch and synchronized GPU completion."""
    import time
    import numpy as np
    import torch
    from transformers import __version__ as transformers_version
    if warmup_batches < 1 or repetitions < 2 or len(dataloader.dataset) < 1:
        raise ValueError("Latency requires nonempty data, warmup >= 1 and repetitions >= 2")
    if not isinstance(dataloader.sampler, torch.utils.data.SequentialSampler):
        raise ValueError("Latency evaluation requires the same sequential inputs for all models")
    device = torch.device(device)

    def synchronize():
        if device.type == "cuda":
            torch.cuda.synchronize(device)

    batch_times, batch_sizes, per_repeat_ms = [], [], []
    lengths = set()
    dtype = None
    with torch.inference_mode():
        warmup_iter = iter(dataloader)
        for _ in range(warmup_batches):
            try:
                batch = next(warmup_iter)
            except StopIteration:
                warmup_iter = iter(dataloader)
                batch = next(warmup_iter)
            inputs = tuple(x.to(device) for x in batch[:3])
            warmup_output = forward_batch(inputs)
            dtype = str(warmup_output.dtype)
            del warmup_output
        synchronize()
        for _ in range(repetitions):
            repeat_ms, repeat_n = 0.0, 0
            for batch in dataloader:
                # Transfers, tokenization, dataloader work and metrics are not timed.
                inputs = tuple(x.to(device) for x in batch[:3])
                synchronize()
                start = time.perf_counter()
                logits = forward_batch(inputs)
                synchronize()
                elapsed_ms = 1000.0 * (time.perf_counter() - start)
                del logits
                n = int(inputs[0].shape[0])
                lengths.add(int(inputs[0].shape[1]))
                batch_times.append(elapsed_ms)
                batch_sizes.append(n)
                repeat_ms += elapsed_ms
                repeat_n += n
            if repeat_n != len(dataloader.dataset):
                raise ValueError("Latency pass did not cover the complete evaluation split")
            per_repeat_ms.append(repeat_ms / repeat_n)
    return {
        "scope": "forward on pretokenized device-resident inputs, including classifier and router if present",
        "excluded": "tokenization, data loading, host-device transfers, loss, metrics, export, FLOPs instrumentation",
        "timer": "perf_counter with CUDA synchronization before and after each forward",
        "warmup_batches": warmup_batches, "repetitions": repetitions,
        "num_examples_per_repeat": len(dataloader.dataset),
        "batch_size": dataloader.batch_size, "observed_batch_sizes": sorted(set(batch_sizes)),
        "observed_sequence_lengths": sorted(lengths),
        "mean_ms_per_batch": float(np.mean(batch_times)),
        "std_ms_per_batch": float(np.std(batch_times, ddof=1)),
        "mean_ms_per_example": float(sum(batch_times) / sum(batch_sizes)),
        "std_ms_per_example_across_repetitions": float(np.std(per_repeat_ms, ddof=1)),
        "ms_per_example_by_repeat": per_repeat_ms,
        "batch_ms_by_repeat": [batch_times[i * len(dataloader):(i + 1) * len(dataloader)]
                              for i in range(repetitions)],
        "device": str(device),
        "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "torch_version": str(torch.__version__), "transformers_version": transformers_version,
        "dtype": dtype,
        "std_note": "Timing variability; distinct from variation between training seeds",
    }


def aligned_test_predictions(routed_jsonl, expert_csv, task):
    import csv
    import json
    import numpy as np
    with open(expert_csv, newline="", encoding="utf-8") as handle:
        expert_rows = list(csv.DictReader(handle))
    expert = {row["example_id"]: row for row in expert_rows}
    if len(expert) != len(expert_rows):
        raise ValueError("Duplicate expert example IDs")
    true, predicted, reference, seen = [], [], [], set()
    with open(routed_jsonl, encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            key = row["example_id"]
            if key in seen or key not in expert:
                raise ValueError(f"Duplicate or unmatched routed example: {key}")
            other = expert[key]
            if row["task"] != task or row["split"] != "test":
                raise ValueError("Bootstrap must use the requested task's test predictions")
            if row["input_sha256"] != other["input_sha256"] or int(row["label"]) != int(other["label"]):
                raise ValueError(f"Input or label mismatch for {key}")
            seen.add(key)
            true.append(int(row["label"]))
            predicted.append(int(row["prediction"]))
            reference.append(int(other["prediction"]))
    if not true or seen != set(expert):
        raise ValueError("Expert and routed prediction sets differ or are empty")
    return tuple(np.asarray(values, dtype=np.int64) for values in (true, predicted, reference))


def paired_f1_bootstrap(labels, predictions, expert_predictions, task, *, resamples=10000, seed=2026):
    """Ordinary IID paired bootstrap via joint-category multinomial counts.

    Resampling counts of (label, routed prediction, expert prediction) is exactly
    equivalent to resampling example indices for F1. Both models share each draw.
    """
    import numpy as np
    if task not in ("sst", "mnli") or resamples < 100:
        raise ValueError("Unknown task or too few bootstrap samples")
    k = 2 if task == "sst" else 3
    arrays = [np.asarray(x, dtype=np.int64) for x in (labels, predictions, expert_predictions)]
    if any(x.ndim != 1 for x in arrays) or len({len(x) for x in arrays}) != 1 or len(arrays[0]) == 0:
        raise ValueError("Predictions must be nonempty aligned vectors")
    if any(np.any((x < 0) | (x >= k)) for x in arrays):
        raise ValueError("Class index outside the fixed label set")
    y, r, e = arrays
    counts = np.bincount(y * k * k + r * k + e, minlength=k ** 3)
    joint = counts.reshape(k, k, k)

    def f1(cm):
        tp = np.diagonal(cm, axis1=-2, axis2=-1)
        denominator = cm.sum(axis=-1) + cm.sum(axis=-2)
        scores = np.divide(2.0 * tp, denominator,
                           out=np.zeros_like(tp, dtype=float), where=denominator != 0)
        return scores[..., 1] if task == "sst" else scores.mean(axis=-1)

    observed = float(f1(joint.sum(axis=2)) - f1(joint.sum(axis=1)))
    rng = np.random.default_rng(seed)
    draws = rng.multinomial(len(y), counts / len(y), size=resamples).reshape(resamples, k, k, k)
    differences = f1(draws.sum(axis=3)) - f1(draws.sum(axis=2))
    lower, upper = np.quantile(differences, [0.025, 0.975])
    # Approximate two-sided bootstrap test: center the bootstrap distribution at H0.
    p_value = float((1 + np.count_nonzero(np.abs(differences - observed) >= abs(observed)))
                    / (resamples + 1))
    return {
        "task": task, "split": "test", "num_examples": len(y),
        "f1_average": "binary" if task == "sst" else "macro",
        "delta_f1": observed, "ci95_low": float(lower), "ci95_high": float(upper),
        "ci_method": "paired IID percentile bootstrap, fixed label set",
        "p_value_two_sided": p_value, "p_value_method": "null-centered paired bootstrap, plus-one correction",
        "resamples": resamples, "bootstrap_seed": seed, "confidence_level": 0.95,
        "scope": "test-example uncertainty for this routed seed against one fixed expert",
        "note": "Not training-seed uncertainty; nonsignificance does not establish equivalence",
    }


In [ ]:
def expert_weight_identity(expert_path):
    """Identify the actual frozen checkpoint, not only its directory name."""
    from pathlib import Path
    import hashlib

    # Local dependency: available before evaluation and after seed cleanup.
    def file_sha256(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    expert_path = Path(expert_path)
    files = [expert_path / "classifier.pt", expert_path / "bert_backbone" / "config.json"]
    files += sorted((expert_path / "bert_backbone").glob("*.safetensors"))
    files += sorted((expert_path / "bert_backbone").glob("pytorch_model*.bin"))
    if len(files) <= 2:
        raise FileNotFoundError(f"No backbone weight files in {expert_path}")
    return {str(path.relative_to(expert_path)): file_sha256(path) for path in files}


def measure_routed_and_fixed_experts():
    import csv
    import json
    import gc
    import torch
    import numpy as np
    from pathlib import Path
    from transformers import AutoModel
    from sklearn.metrics import confusion_matrix
    first_manifest_path = EXPERIMENT_RESULTS_PATH / f"{EXPERIMENT_TIMESTAMP}_seed_1" / "comparison_manifest.json"
    previous = json.loads(first_manifest_path.read_text(encoding="utf-8")) if SEED_INDEX != 1 else None
    fixed_experts = {}
    routed_model.eval()
    for task, loader, expert_path in (
        ("sst", sst_test_loader, SST_EXPERT_PATH),
        ("mnli", mnli_test_loader, MNLI_EXPERT_PATH),
    ):
        # Routed and expert receive the exact same CPU tensor dataset, padding and batch sizes.
        result = evaluation_results[f"{task}_test"]
        result["latency"] = benchmark_inference(
            lambda inputs: routed_model(input_ids=inputs[0], attention_mask=inputs[1],
                                        token_type_ids=inputs[2], task=task, tau=TAU_EVAL,
                                        efficient_inference=True, return_routing=False),
            loader, DEVICE, warmup_batches=LATENCY_WARMUP_BATCHES, repetitions=LATENCY_REPETITIONS)
        identity = expert_weight_identity(expert_path)
        prediction_path = EXPERIMENT_RESULTS_PATH / f"expert_{task}_predictions.csv"
        if previous is None:
            backbone = AutoModel.from_pretrained(expert_path / "bert_backbone").to(DEVICE).eval()
            head = ClassifierHead(input_dim=HIDDEN_SIZE, hidden_dim=HEAD_HIDDEN_SIZE,
                                  num_classes=2 if task == "sst" else 3,
                                  dropout_prob=DROPOUT_PROBABILITY).to(DEVICE)
            head.load_state_dict(torch.load(expert_path / "classifier.pt", map_location=DEVICE,
                                            weights_only=True), strict=True)
            head.eval()

            def forward_expert(inputs):
                output = backbone(input_ids=inputs[0], attention_mask=inputs[1], token_type_ids=inputs[2])
                return head(output.pooler_output)

            cost = new_inference_cost()
            true, predicted = [], []
            partial = prediction_path.with_suffix(".csv.partial")
            with partial.open("w", newline="", encoding="utf-8") as handle, torch.no_grad():
                writer = csv.DictWriter(handle, fieldnames=["example_id", "input_sha256", "label", "prediction"])
                writer.writeheader()
                index = 0
                for batch in loader:
                    inputs = tuple(x.to(DEVICE) for x in batch[:3])
                    with RoutedFlopCounter() as counter:
                        logits = forward_expert(inputs)
                    add_inference_cost(cost, counter, "original_expert", inputs[0])
                    predictions = logits.argmax(dim=-1).cpu().tolist()
                    for ids, mask, types, label, prediction in zip(
                            batch[0].tolist(), batch[1].tolist(), batch[2].tolist(), batch[3].tolist(), predictions):
                        writer.writerow({"example_id": f"{task}:test:{index}",
                                         "input_sha256": input_fingerprint(ids, mask, types),
                                         "label": label, "prediction": prediction})
                        true.append(label)
                        predicted.append(prediction)
                        index += 1
            if index != len(loader.dataset):
                raise ValueError("Incomplete expert prediction export")
            partial.replace(prediction_path)
            metrics = compute_metrics(true, predicted, task)
            metrics["num_samples"] = index
            inference_cost = finish_inference_cost(cost, index)
            inference_cost["scope"] = "original expert: complete backbone and original classifier; no router"
            inference_cost["convention"].pop("router", None)
            fixed_experts[task] = {
                "metrics": metrics,
                "confusion_matrix": confusion_matrix(true, predicted, labels=list(range(2 if task == "sst" else 3))).tolist(),
                "inference_cost": inference_cost,
                "latency": benchmark_inference(forward_expert, loader, DEVICE,
                                               warmup_batches=LATENCY_WARMUP_BATCHES,
                                               repetitions=LATENCY_REPETITIONS),
                "predictions_csv": str(prediction_path), "predictions_sha256": file_sha256(prediction_path),
                "checkpoint_directory": str(expert_path), "checkpoint_sha256": identity,
                "reference_seed_index": 1, "additional_training": False,
            }
            del head, backbone, logits
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        else:
            fixed_experts[task] = previous["fixed_expert_test"][task]
            if fixed_experts[task]["checkpoint_sha256"] != identity:
                raise ValueError("Expert weights changed between routed seeds")
            if fixed_experts[task]["predictions_sha256"] != file_sha256(prediction_path):
                raise ValueError("Shared expert predictions changed between routed seeds")
        y, r, e = aligned_test_predictions(result["per_input"]["jsonl_path"], prediction_path, task)
        result["bootstrap_vs_expert"] = paired_f1_bootstrap(
            y, r, e, task, resamples=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED)
        print(f"{task.upper()} test: F1 ({result['metrics']['f1_average']})={result['metrics']['f1']:.6f}; "
              f"accuracy={result['metrics']['accuracy']:.6f}; "
              f"latency={result['latency']['mean_ms_per_example']:.3f} ms/example")
    comparison_manifest["fixed_expert_test"] = fixed_experts
    comparison_manifest["measurement_status"] = "completed"
    comparison_manifest["criteria"]["uncertainty_analysis"] = True
    comparison_manifest["criteria"]["uncertainty_scope"] = "paired test-example bootstrap per routed seed; experts fixed"
    comparison_manifest["experiment_group"] = experiment_group_metadata()
    report["fixed_expert_test"] = fixed_experts
    report["experiment_group"] = experiment_group_metadata()
    report["detailed_results"]["splits"] = evaluation_results
    for task in ("sst", "mnli"):
        report["summary"]["test"][task].update({
            "f1_average": "binary" if task == "sst" else "macro",
            "latency": evaluation_results[f"{task}_test"]["latency"],
            "bootstrap_vs_expert": evaluation_results[f"{task}_test"]["bootstrap_vs_expert"],
        })
    training_summary["experiment_group"] = experiment_group_metadata()
    for path, data in ((MANIFEST_PATH, comparison_manifest), (REPORT_JSON_PATH, report),
                       (TRAINING_SUMMARY_PATH, training_summary)):
        atomic_save_json(path, data)


def experiment_group_metadata():
    return {
        "timestamp": EXPERIMENT_TIMESTAMP, "seed_index": SEED_INDEX, "seed": SEED,
        "seeds": list(RUN_SEEDS), "num_seeds_planned": len(RUN_SEEDS),
        "results_directory": str(EXPERIMENT_RESULTS_PATH),
        "seed_directories": [f"{EXPERIMENT_TIMESTAMP}_seed_{i}" for i in range(1, len(RUN_SEEDS) + 1)],
        "statistical_summary_csv": str(EXPERIMENT_RESULTS_PATH / "statistical_summary.csv"),
        "scope": "three independently initialized routed trainings on the same fixed experts and data",
    }


def save_three_seed_summary(experiment_results_path, experiment_timestamp, seeds):
    import csv
    import json
    import math
    import numpy as np
    import pandas as pd
    from pathlib import Path
    parent = Path(experiment_results_path)
    records = []
    for index, seed in enumerate(seeds, 1):
        folder = parent / f"{experiment_timestamp}_seed_{index}"
        manifest_path = folder / "comparison_manifest.json"
        item = json.loads(manifest_path.read_text(encoding="utf-8"))
        if item.get("measurement_status") != "completed" or item["experiment_group"]["seed"] != seed:
            raise ValueError(f"Incomplete or incorrectly labeled seed: {folder}")
        records.append((folder, manifest_path, item))
    rows, aggregates = [], {}

    def row_for(task, kind, seed, item, expert=None):
        metrics = item["metrics"]
        cost = item["inference_cost"]
        latency = item["latency"]
        flops = cost.get("flops_per_example")
        coverage = cost.get("coverage", {})
        if flops is None and set(coverage.get("uncovered_operators", [])) == {"aten.constant_pad_nd"}:
            flops = coverage["counted_flops"] / cost["num_examples"]
        boot = item.get("bootstrap_vs_expert", {})
        row = {"task": task, "row_type": kind, "seed": seed,
               "num_examples": metrics["num_samples"], "f1_average": metrics["f1_average"],
               "f1": metrics["f1"], "accuracy": metrics["accuracy"],
               "flops_per_example": flops, "flops_coverage": coverage.get("status"),
               "latency_ms_per_example": latency["mean_ms_per_example"],
               "latency_repeat_std_ms_per_example": latency["std_ms_per_example_across_repetitions"],
               "latency_mean_ms_per_batch": latency["mean_ms_per_batch"],
               "latency_std_ms_per_batch": latency["std_ms_per_batch"],
               "delta_f1_vs_expert": boot.get("delta_f1"),
               "bootstrap_ci95_low": boot.get("ci95_low"), "bootstrap_ci95_high": boot.get("ci95_high"),
               "bootstrap_p_value_two_sided": boot.get("p_value_two_sided")}
        return row

    for task in ("sst", "mnli"):
        reference = records[0][2]["fixed_expert_test"][task]
        task_rows = []
        for folder, _, item in records:
            other = item["fixed_expert_test"][task]
            if (other["checkpoint_sha256"] != reference["checkpoint_sha256"]
                    or other["predictions_sha256"] != reference["predictions_sha256"]):
                raise ValueError("Different expert references cannot be pooled")
            current = row_for(task, "routed", item["experiment_group"]["seed"], item["splits"][f"{task}_test"])
            current["run_directory"] = folder.name
            task_rows.append(current)
            rows.append(current)
        rows.append(row_for(task, "fixed_expert", None, reference))
        aggregate = {"task": task, "row_type": "routed_mean", "seed": None,
                     "num_examples": task_rows[0]["num_examples"], "f1_average": task_rows[0]["f1_average"],
                     "num_seeds": len(seeds), "flops_coverage": "see individual seeds"}
        for key in ("f1", "accuracy", "flops_per_example", "latency_ms_per_example",
                    "latency_mean_ms_per_batch", "delta_f1_vs_expert"):
            values = [row[key] for row in task_rows]
            valid = all(isinstance(x, (int, float)) and math.isfinite(x) for x in values)
            aggregate[key] = float(np.mean(values)) if valid else None
            aggregate[key + "_std_across_seeds"] = float(np.std(values, ddof=1)) if valid else None
        aggregate["std_definition"] = "sample standard deviation across three routed training seeds, ddof=1"
        rows.append(aggregate)
        aggregates[task] = aggregate
    fields = list(dict.fromkeys(key for row in rows for key in row))
    destination = parent / "statistical_summary.csv"
    partial = destination.with_suffix(".csv.partial")
    with partial.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    partial.replace(destination)
    for folder, manifest_path, item in records:
        item["seed_aggregate"] = {
            "status": "completed", "num_seeds": len(seeds), "seeds": list(seeds),
            "tasks": aggregates, "summary_csv": str(destination),
            "note": "Bootstrap CIs and p-values apply to individual seeds, not to the across-seed mean",
        }
        atomic_save_json(manifest_path, item)
        report_path = folder / Path(item["report_path"]).name
        saved_report = json.loads(report_path.read_text(encoding="utf-8"))
        saved_report["seed_aggregate"] = item["seed_aggregate"]
        atomic_save_json(report_path, saved_report)
    print("\nThree routed seeds on fixed experts — test results")
    display_columns = ["task", "row_type", "seed", "f1_average", "f1", "f1_std_across_seeds",
                       "accuracy", "accuracy_std_across_seeds", "flops_per_example",
                       "latency_ms_per_example", "delta_f1_vs_expert", "bootstrap_ci95_low",
                       "bootstrap_ci95_high", "bootstrap_p_value_two_sided"]
    frame = pd.DataFrame(rows).reindex(columns=display_columns)
    print(frame.to_string(index=False))
    for task, value in aggregates.items():
        print(f"{task.upper()} F1 ({value['f1_average']}): "
              f"{100*value['f1']:.3f} +/- {100*value['f1_std_across_seeds']:.3f} percentage points")
    print("Saved:", destination)


In [ ]:
def atomic_save_file(destination, writer):
    """Commit a complete file; an interrupted write leaves the previous file intact."""
    import os
    from pathlib import Path
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".partial")
    with temporary.open("wb") as handle:
        writer(handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, destination)
    if os.name == "posix":
        fd = os.open(str(destination.parent), os.O_RDONLY | os.O_DIRECTORY)
        try:
            os.fsync(fd)
        finally:
            os.close(fd)


def atomic_save_json(destination, data):
    import json
    atomic_save_file(destination, lambda handle: handle.write(
        json.dumps(data, indent=2, ensure_ascii=False).encode("utf-8")))


def execution_lock(root):
    """An OS lock is released even after kill/reboot; never start two writers."""
    import os
    from pathlib import Path
    from contextlib import contextmanager

    @contextmanager
    def locked():
        root_path = Path(root)
        root_path.mkdir(parents=True, exist_ok=True)
        with (root_path / ".execution.lock").open("a+b") as handle:
            if os.name == "posix":
                import fcntl
                try:
                    fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
                except BlockingIOError as exc:
                    raise RuntimeError("Un'altra esecuzione RoutedModel e' gia' attiva.") from exc
            else:
                import msvcrt
                handle.seek(0)
                if not handle.read(1):
                    handle.write(b"0")
                    handle.flush()
                handle.seek(0)
                msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
            try:
                yield
            finally:
                if os.name == "posix":
                    fcntl.flock(handle, fcntl.LOCK_UN)
                else:
                    handle.seek(0)
                    msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
    return locked()


def choose_execution(root, seeds):
    """Inspect the newest timestamp only. Completed => new; incomplete => resume."""
    import json
    import re
    from pathlib import Path
    from datetime import datetime
    root = Path(root)
    results = root / "Results"
    results.mkdir(parents=True, exist_ok=True)
    folders = sorted(p for p in results.iterdir() if p.is_dir()
                     and re.fullmatch(r"\d{8}_\d{6}(?:_\d{6})?", p.name))
    state = None
    if folders:
        parent = folders[-1]
        manifest = parent / "execution_manifest.json"
        if manifest.exists():
            state = json.loads(manifest.read_text(encoding="utf-8"))
            if state.get("schema_version") != 1:
                raise ValueError(f"Manifest di ripresa non supportato: {manifest}")
            if state.get("timestamp") != parent.name:
                raise ValueError("Il timestamp del manifest non corrisponde alla cartella")
        else:
            # Older folders have per-seed manifests only. No hard-coded run is recovered.
            entries = {}
            for index, seed in enumerate(seeds, 1):
                folder = parent / f"{parent.name}_seed_{index}"
                path = folder / "comparison_manifest.json"
                complete = False
                if path.exists():
                    item = json.loads(path.read_text(encoding="utf-8"))
                    complete = (item.get("measurement_status") == "completed"
                                and item.get("status") == "evaluation_completed"
                                and item.get("experiment_group", {}).get("seed") == seed)
                entries[str(index)] = {"seed": seed, "status": "completed" if complete else "pending",
                                       "epochs_completed": 0}
            finished = (all(x["status"] == "completed" for x in entries.values())
                        and (parent / "statistical_summary.csv").is_file())
            state = {"schema_version": 1, "timestamp": parent.name, "seeds": list(seeds),
                     "status": "completed" if finished else "in_progress", "seed_states": entries}
        if state["status"] == "completed":
            state = None
        elif state["seeds"] != list(seeds):
            raise ValueError("L'ultima esecuzione incompleta usa seed diversi: ripristinare RUN_SEEDS.")
    if state is None:
        stamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S_%f")
        parent = results / stamp
        parent.mkdir(exist_ok=False)
        state = {"schema_version": 1, "timestamp": stamp, "seeds": list(seeds),
                 "status": "in_progress", "seed_states": {
                     str(i): {"seed": seed, "status": "pending", "epochs_completed": 0}
                     for i, seed in enumerate(seeds, 1)}}
        print("Nuova esecuzione:", stamp)
    else:
        print("Ripresa automatica dell'ultima esecuzione:", state["timestamp"])
    path = parent / "execution_manifest.json"
    atomic_save_json(path, state)
    return path, state


def record_execution_seed(status, epochs_completed=None):
    from datetime import datetime
    row = EXECUTION_STATE["seed_states"][str(SEED_INDEX)]
    row["status"] = status
    if epochs_completed is not None:
        row["epochs_completed"] = int(epochs_completed)
    row["updated_at"] = datetime.now().astimezone().isoformat(timespec="seconds")
    atomic_save_json(EXECUTION_MANIFEST_PATH, EXECUTION_STATE)


def mutable_model_state(model):
    """Best-model snapshots need only trainable parameters and buffers; encoders are frozen."""
    names = {name for name, parameter in model.named_parameters() if parameter.requires_grad}
    names.update(name for name, _ in model.named_buffers())
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()
            if name in names}


def make_resume_contract():
    import hashlib
    from pathlib import Path
    # Exclude runtime identifiers, paths and display options, retain training settings.
    ignored = {"RUN_STATE_NAMES", "RUN_STARTED_AT", "RUN_TIMESTAMP", "EXPERIMENT_TIMESTAMP",
               "ROUTED_BASELINES_DIRECTORY", "SHOW_BATCH_PROGRESS", "PIN_MEMORY"}
    parameters = {key: value for key, value in experiment_config["parameters"].items()
                  if key not in ignored and not key.endswith("PATH")}
    data = {}
    for task in ("SST", "MNLI"):
        for field in ("X", "Y"):
            for split in ("TRAIN", "VAL", "TEST"):
                key = f"{task}_{field}_{split}_PATH"
                digest = hashlib.sha256()
                with Path(globals()[key]).open("rb") as handle:
                    for block in iter(lambda: handle.read(1024 * 1024), b""):
                        digest.update(block)
                data[key] = digest.hexdigest()
    return {"parameters": parameters, "datasets": data,
            "experts": {"sst": expert_weight_identity(SST_EXPERT_PATH),
                        "mnli": expert_weight_identity(MNLI_EXPERT_PATH)},
            "architecture": experiment_config["architecture"],
            "input_preparation": experiment_config["input_preparation"],
            "optimizer": experiment_config["optimizer"],
            "scheduler": experiment_config["scheduler"]}


def save_epoch_checkpoint(contract, best_states):
    import random
    import time
    import numpy as np
    import torch
    state_names = (
        "training_history", "routing_history", "training_cost_by_task", "best_epoch",
        "best_val_f1_avg", "best_routing_stats", "best_validation_metrics",
        "best_quality_checkpoint", "best_efficiency_checkpoint", "validation_references",
        "experiment_config", "RUN_STARTED_AT",
    )
    payload = {
        "schema_version": 1, "timestamp": EXPERIMENT_TIMESTAMP, "seed": SEED,
        "epochs_completed": len(training_history), "contract": contract,
        "model": routed_model.state_dict(), "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(), "early_stopping": vars(early_stopping).copy(),
        "best_states": best_states, "training_state": {key: globals()[key] for key in state_names},
        "training_time_seconds": time.time() - start_time,
        "rng": {"python": random.getstate(), "numpy": np.random.get_state(),
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []},
    }
    # Do not overwrite the last good checkpoint until serialization and fsync succeed.
    atomic_save_file(MODELS_PATH / "training_resume.pt", lambda handle: torch.save(payload, handle))
    record_execution_seed("training_completed" if len(training_history) == EPOCHS else "training",
                          len(training_history))


def restore_epoch_checkpoint(contract):
    import random
    import time
    import numpy as np
    import torch
    path = MODELS_PATH / "training_resume.pt"
    if not path.is_file():
        if EXECUTION_STATE["seed_states"][str(SEED_INDEX)].get("epochs_completed", 0) > 0:
            raise FileNotFoundError(f"Checkpoint di ripresa mancante: {path}")
        return {}
    # Local checkpoints contain NumPy/Python RNG and training dictionaries, not only weights.
    saved = torch.load(path, map_location="cpu", weights_only=False)
    if (saved.get("schema_version") != 1 or saved["timestamp"] != EXPERIMENT_TIMESTAMP
            or saved["seed"] != SEED):
        raise ValueError("Checkpoint appartenente a un'altra esecuzione o seed")
    if saved["contract"] != contract:
        raise ValueError("Configurazione, dati o esperti modificati: ripristinarli prima della ripresa.")
    completed = saved["epochs_completed"]
    if not 0 <= completed <= EPOCHS or len(saved["training_state"]["training_history"]) != completed:
        raise ValueError("Avanzamento del checkpoint incoerente")
    routed_model.load_state_dict(saved["model"], strict=True)
    optimizer.load_state_dict(saved["optimizer"])
    scheduler.load_state_dict(saved["scheduler"])
    early_stopping.__dict__.update(saved["early_stopping"])
    globals().update(saved["training_state"])
    globals()["start_time"] = time.time() - saved["training_time_seconds"]
    random.setstate(saved["rng"]["python"])
    np.random.set_state(saved["rng"]["numpy"])
    torch.set_rng_state(saved["rng"]["torch"])
    if saved["rng"]["cuda"]:
        if not torch.cuda.is_available() or len(saved["rng"]["cuda"]) != torch.cuda.device_count():
            raise RuntimeError("Ripresa richiesta con la stessa disponibilita' CUDA del checkpoint")
        torch.cuda.set_rng_state_all(saved["rng"]["cuda"])
    print(f"Seed {SEED}: ripristinate {completed}/{EPOCHS} epoche; "
          + (f"riparto dall'epoca {completed + 1}." if completed < EPOCHS else "riprendo la valutazione."))
    return saved["best_states"]


def export_committed_best_models(best_states):
    """Rebuild exports from the committed epoch, even if a previous export was interrupted."""
    import torch
    current = mutable_model_state(routed_model)
    try:
        for kind, destination in (("quality", ROUTED_MODEL_SAVE_PATH),
                                  ("efficiency", BEST_EFFICIENCY_MODEL_PATH)):
            if kind in best_states:
                routed_model.load_state_dict(best_states[kind], strict=False)
                atomic_save_file(destination, lambda handle: torch.save(routed_model, handle))
    finally:
        routed_model.load_state_dict(current, strict=False)


In [ ]:
def initialize_paths():
    global BASELINES_PATH, DATASET_PATH, EXPERIMENT_MODELS_PATH, EXPERIMENT_NAME, EXPERIMENT_RESULTS_PATH, MERGED_MODE, MIN_INFERENCE_FLOPS_SAVING, MNLI_BACKBONE_PATH
    global MNLI_DATASET_PATH, MNLI_EXPERT_PATH, MNLI_X_TEST_PATH, MNLI_X_TRAIN_PATH, MNLI_X_VAL_PATH, MNLI_Y_TEST_PATH, MNLI_Y_TRAIN_PATH, MNLI_Y_VAL_PATH
    global MODELS_PATH, MODEL_NAME, MODEL_VARIANT, Path, QUALITY_TOLERANCE_ABSOLUTE, RESULTS_PATH, ROUTED_MODEL_PATH, ROUTED_MODEL_SAVE_PATH
    global ROUTING_ANALYSIS_PATH, ROUTING_AVAILABILITY, RUN_STARTED_AT, RUN_TIMESTAMP, SST_BACKBONE_PATH, SST_DATASET_PATH, SST_EXPERT_PATH, SST_X_TEST_PATH
    global SST_X_TRAIN_PATH, SST_X_VAL_PATH, SST_Y_TEST_PATH, SST_Y_TRAIN_PATH, SST_Y_VAL_PATH, TRAINED_EXPERTS_PATH, TRAINING_HISTORY_PATH, TRAINING_SUMMARY_PATH
    global datetime, depth, merged_choices_for_layer
    # ============================================================
    # INITIALIZATION: PATHS AND FOLDERS
    # ============================================================

    from pathlib import Path
    from datetime import datetime

    RUN_STARTED_AT = datetime.now().astimezone().isoformat(timespec="microseconds")
    RUN_TIMESTAMP = f"{EXPERIMENT_TIMESTAMP}_seed_{SEED_INDEX}"

    # ============================================================
    # MAIN PATHS
    # ============================================================

    # Explicit paths on the remote GPU machine.
    MODEL_NAME = "RoutedModel"
    MODEL_VARIANT = "MergedFirstLayer"
    # Choices of encoder layers only; shared embeddings and pooler are unchanged.
    MERGED_MODE = "impose"  # "off", "auto", or "impose"


    def merged_choices_for_layer(mode, layer_idx):
        if mode not in ("off", "auto", "impose"):
            raise ValueError("MERGED_MODE must be off, auto, or impose")
        if layer_idx < 0:
            raise ValueError("Layer index must be non-negative")
        if layer_idx > 0 or mode == "off":
            return ["SST", "MNLI", "IDENTITY"]
        if mode == "impose":
            return ["MERGED"]
        return ["SST", "MNLI", "MERGED", "IDENTITY"]


    ROUTING_AVAILABILITY = {
        "num_routing_layers": 12,
        "merged_mode": MERGED_MODE,
        "merged_available_layers": [] if MERGED_MODE == "off" else [1],
        "first_layer_forced": MERGED_MODE == "impose",
        "available_choices_by_layer": [
            merged_choices_for_layer(MERGED_MODE, depth) for depth in range(12)
        ],
        "note": "Unavailable encoder choices are masked before routing and not executed",
    }

    BASELINES_PATH = Path(ROUTED_BASELINES_DIRECTORY)

    ROUTED_MODEL_PATH = BASELINES_PATH / MODEL_NAME
    DATASET_PATH = BASELINES_PATH.parent / "Preprocessing" / "Dataset" / "ProcessedDB"

    TRAINED_EXPERTS_PATH = BASELINES_PATH / "TrainingOfUncasedModel" / "Models"

    # ============================================================
    # OUTPUT FOLDERS
    # ============================================================

    # Model identity; output folders are isolated by run timestamp.
    EXPERIMENT_NAME = f"{MODEL_NAME}_{MODEL_VARIANT}"
    EXPERIMENT_RESULTS_PATH = ROUTED_MODEL_PATH / "Results" / EXPERIMENT_TIMESTAMP
    EXPERIMENT_MODELS_PATH = ROUTED_MODEL_PATH / "Models" / EXPERIMENT_TIMESTAMP
    RESULTS_PATH = EXPERIMENT_RESULTS_PATH / RUN_TIMESTAMP
    MODELS_PATH = EXPERIMENT_MODELS_PATH / RUN_TIMESTAMP

    # Comparison criteria only: these do not change training or checkpoint selection.
    # Absolute score tolerance: e.g. 0.90 -> 0.89. FLOP saving must be strictly > 0.01.
    QUALITY_TOLERANCE_ABSOLUTE = 0.01
    MIN_INFERENCE_FLOPS_SAVING = 0.01

    RESULTS_PATH.mkdir(parents=True, exist_ok=True)
    MODELS_PATH.mkdir(parents=True, exist_ok=True)

    # ============================================================
    # DATASET PATHS
    # ============================================================

    SST_DATASET_PATH = DATASET_PATH / "SST"
    MNLI_DATASET_PATH = DATASET_PATH / "MNLI"

    # SST
    SST_X_TRAIN_PATH = SST_DATASET_PATH / "X_train.parquet"
    SST_Y_TRAIN_PATH = SST_DATASET_PATH / "y_train.parquet"
    SST_X_VAL_PATH   = SST_DATASET_PATH / "X_val.parquet"
    SST_Y_VAL_PATH   = SST_DATASET_PATH / "y_val.parquet"
    SST_X_TEST_PATH  = SST_DATASET_PATH / "X_test.parquet"
    SST_Y_TEST_PATH  = SST_DATASET_PATH / "y_test.parquet"

    # MNLI
    MNLI_X_TRAIN_PATH = MNLI_DATASET_PATH / "X_train.parquet"
    MNLI_Y_TRAIN_PATH = MNLI_DATASET_PATH / "y_train.parquet"
    MNLI_X_VAL_PATH   = MNLI_DATASET_PATH / "X_val.parquet"
    MNLI_Y_VAL_PATH   = MNLI_DATASET_PATH / "y_val.parquet"
    MNLI_X_TEST_PATH  = MNLI_DATASET_PATH / "X_test.parquet"
    MNLI_Y_TEST_PATH  = MNLI_DATASET_PATH / "y_test.parquet"

    # ============================================================
    # EXPERT MODEL PATHS
    # ============================================================

    SST_EXPERT_PATH = TRAINED_EXPERTS_PATH / "sst_bert_base_uncased"
    MNLI_EXPERT_PATH = TRAINED_EXPERTS_PATH / "mnli_bert_base_uncased"

    SST_BACKBONE_PATH = SST_EXPERT_PATH / "bert_backbone"
    MNLI_BACKBONE_PATH = MNLI_EXPERT_PATH / "bert_backbone"


    # ============================================================
    # ROUTED MODEL OUTPUT PATHS
    # ============================================================

    ROUTED_MODEL_SAVE_PATH = MODELS_PATH / "routed_model.pt"

    TRAINING_HISTORY_PATH = RESULTS_PATH / "routed_training_history.json"
    TRAINING_SUMMARY_PATH = RESULTS_PATH / "routed_training_summary.json"
    ROUTING_ANALYSIS_PATH = RESULTS_PATH / "routing_analysis.json"

    # ============================================================
    # PRINT CHECK
    # ============================================================

    print("Variant:", MODEL_VARIANT, "| merged mode:", MERGED_MODE)
    print(f"{MODEL_NAME} path:", ROUTED_MODEL_PATH)
    print("Results path:", RESULTS_PATH)
    print("Models path:", MODELS_PATH)
    print("Dataset path:", DATASET_PATH)
    print("SST expert path:", SST_EXPERT_PATH)
    print("MNLI expert path:", MNLI_EXPERT_PATH)

In [ ]:
def load_datasets():
    global X_mnli_test, X_mnli_train, X_mnli_val, X_sst_test, X_sst_train, X_sst_val, pd, x_mnli_test
    global x_mnli_train, x_mnli_val, x_sst_test, x_sst_train, x_sst_val, y_mnli_test, y_mnli_train, y_mnli_val
    global y_sst_test, y_sst_train, y_sst_val
    # ============================================================
    # LOAD DATASETS
    # ============================================================

    import pandas as pd

    # ============================================================
    # LOAD SST
    # ============================================================

    X_sst_train = pd.read_parquet(SST_X_TRAIN_PATH)
    y_sst_train = pd.read_parquet(SST_Y_TRAIN_PATH)

    X_sst_val = pd.read_parquet(SST_X_VAL_PATH)
    y_sst_val = pd.read_parquet(SST_Y_VAL_PATH)

    X_sst_test = pd.read_parquet(SST_X_TEST_PATH)
    y_sst_test = pd.read_parquet(SST_Y_TEST_PATH)

    # ============================================================
    # LOAD MNLI
    # ============================================================

    X_mnli_train = pd.read_parquet(MNLI_X_TRAIN_PATH)
    y_mnli_train = pd.read_parquet(MNLI_Y_TRAIN_PATH)

    X_mnli_val = pd.read_parquet(MNLI_X_VAL_PATH)
    y_mnli_val = pd.read_parquet(MNLI_Y_VAL_PATH)

    X_mnli_test = pd.read_parquet(MNLI_X_TEST_PATH)
    y_mnli_test = pd.read_parquet(MNLI_Y_TEST_PATH)

    # ============================================================
    # CONVERT TO LISTS
    # ============================================================

    x_sst_train = X_sst_train["text"].tolist()
    x_sst_val   = X_sst_val["text"].tolist()
    x_sst_test  = X_sst_test["text"].tolist()

    y_sst_train = y_sst_train["label"].astype(int).tolist()
    y_sst_val   = y_sst_val["label"].astype(int).tolist()
    y_sst_test  = y_sst_test["label"].astype(int).tolist()

    x_mnli_train = X_mnli_train["text"].tolist()
    x_mnli_val   = X_mnli_val["text"].tolist()
    x_mnli_test  = X_mnli_test["text"].tolist()

    y_mnli_train = y_mnli_train["label"].astype(int).tolist()
    y_mnli_val   = y_mnli_val["label"].astype(int).tolist()
    y_mnli_test  = y_mnli_test["label"].astype(int).tolist()

    # ============================================================
    # PRINT DATASET INFO
    # ============================================================

    print("SST")
    print("Train:", len(x_sst_train), len(y_sst_train))
    print("Val:  ", len(x_sst_val), len(y_sst_val))
    print("Test: ", len(x_sst_test), len(y_sst_test))
    print("Labels train:", pd.Series(y_sst_train).value_counts().sort_index().to_dict())

    print("\nMNLI")
    print("Train:", len(x_mnli_train), len(y_mnli_train))
    print("Val:  ", len(x_mnli_val), len(y_mnli_val))
    print("Test: ", len(x_mnli_test), len(y_mnli_test))
    print("Labels train:", pd.Series(y_mnli_train).value_counts().sort_index().to_dict())

In [ ]:
def select_dataset_subset():
    global DEBUG_FRACTION, pd, take_debug_subset, train_test_split, x_mnli_test_run, x_mnli_train_run, x_mnli_val_run, x_sst_test_run
    global x_sst_train_run, x_sst_val_run, y_mnli_test_run, y_mnli_train_run, y_mnli_val_run, y_sst_test_run, y_sst_train_run, y_sst_val_run
    # ============================================================
    # DEBUG SUBSET
    # ============================================================

    from sklearn.model_selection import train_test_split
    import pandas as pd

    # ============================================================
    # DEBUG PARAMETER
    # ============================================================

    DEBUG_FRACTION = 1   # Change this value: 0.10 = 10%, 0.20 = 20%, 1.00 = full dataset
    # SEED is assigned by run_three_seeds before model creation.

    # ============================================================
    # SUBSET FUNCTION
    # ============================================================

    def take_debug_subset(x, y, fraction, seed=42):
        if fraction >= 1.0:
            return x, y

        x_subset, _, y_subset, _ = train_test_split(
            x,
            y,
            train_size=fraction,
            random_state=seed,
            stratify=y
        )

        return x_subset, y_subset

    # ============================================================
    # APPLY DEBUG SUBSET
    # ============================================================

    x_sst_train_run, y_sst_train_run = take_debug_subset(x_sst_train, y_sst_train, DEBUG_FRACTION, DATA_SEED)
    x_sst_val_run,   y_sst_val_run   = take_debug_subset(x_sst_val,   y_sst_val,   DEBUG_FRACTION, DATA_SEED)
    x_sst_test_run,  y_sst_test_run  = take_debug_subset(x_sst_test,  y_sst_test,  DEBUG_FRACTION, DATA_SEED)

    x_mnli_train_run, y_mnli_train_run = take_debug_subset(x_mnli_train, y_mnli_train, DEBUG_FRACTION, DATA_SEED)
    x_mnli_val_run,   y_mnli_val_run   = take_debug_subset(x_mnli_val,   y_mnli_val,   DEBUG_FRACTION, DATA_SEED)
    x_mnli_test_run,  y_mnli_test_run  = take_debug_subset(x_mnli_test,  y_mnli_test,  DEBUG_FRACTION, DATA_SEED)

    # ============================================================
    # PRINT DEBUG INFO
    # ============================================================

    print("DEBUG FRACTION:", DEBUG_FRACTION)

    print("\nSST")
    print("Train:", len(x_sst_train_run), pd.Series(y_sst_train_run).value_counts().sort_index().to_dict())
    print("Val:  ", len(x_sst_val_run),   pd.Series(y_sst_val_run).value_counts().sort_index().to_dict())
    print("Test: ", len(x_sst_test_run),  pd.Series(y_sst_test_run).value_counts().sort_index().to_dict())

    print("\nMNLI")
    print("Train:", len(x_mnli_train_run), pd.Series(y_mnli_train_run).value_counts().sort_index().to_dict())
    print("Val:  ", len(x_mnli_val_run),   pd.Series(y_mnli_val_run).value_counts().sort_index().to_dict())
    print("Test: ", len(x_mnli_test_run),  pd.Series(y_mnli_test_run).value_counts().sort_index().to_dict())

In [ ]:
def load_models_and_heads():
    global AutoModel, AutoTokenizer, ClassifierHead, DEVICE, DROPOUT_PROBABILITY, HEAD_HIDDEN_SIZE, HIDDEN_SIZE, MERGE_ALPHA_SST
    global MERGE_BETA_MNLI, TOKENIZER_NAME, count_trainable_params, deepcopy, merge_cost, merge_float_elements, merge_float_tensors, merged_backbone
    global merged_config, mnli_backbone, mnli_head, mnli_param, mnli_state, name, nn, param
    global sst_backbone, sst_head, sst_param, sst_state, target, tokenizer, torch
    # ============================================================
    # LOAD MODELS, TOKENIZER, AND CLASSIFIERS
    # ============================================================

    import torch
    import torch.nn as nn
    from transformers import AutoTokenizer, AutoModel

    MERGE_ALPHA_SST = 0.5
    MERGE_BETA_MNLI = 1.0 - MERGE_ALPHA_SST

    assert 0.0 <= MERGE_ALPHA_SST <= 1.0
    assert abs(MERGE_ALPHA_SST + MERGE_BETA_MNLI - 1.0) < 1e-8

    # ============================================================
    # DEVICE
    # ============================================================

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

    # ============================================================
    # TOKENIZER
    # ============================================================

    TOKENIZER_NAME = "google-bert/bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

    # ============================================================
    # LOAD FROZEN EXPERT BACKBONES
    # ============================================================

    sst_backbone = AutoModel.from_pretrained(SST_BACKBONE_PATH).to(DEVICE)
    mnli_backbone = AutoModel.from_pretrained(MNLI_BACKBONE_PATH).to(DEVICE)

    sst_backbone.eval()
    mnli_backbone.eval()

    for param in sst_backbone.parameters():
        param.requires_grad = False

    for param in mnli_backbone.parameters():
        param.requires_grad = False

    # ============================================================
    # CREATE ONLY MERGED EMBEDDINGS, FIRST ENCODER LAYER AND POOLER
    # ============================================================

    from copy import deepcopy

    # Never instantiate the eleven unused merged layers, even temporarily on GPU.
    # Temporary initialization consumes no training RNG; every stored tensor is
    # replaced below by the same weighted expert tensors as in the full variant.
    merged_config = deepcopy(sst_backbone.config)
    merged_config.num_hidden_layers = 1
    with torch.random.fork_rng(devices=[]):
        merged_backbone = AutoModel.from_config(merged_config).to(DEVICE)

    sst_state = sst_backbone.state_dict()
    mnli_state = mnli_backbone.state_dict()
    merge_float_elements = 0
    merge_float_tensors = 0

    with torch.no_grad():
        for name, target in merged_backbone.state_dict().items():
            if not (name.startswith("embeddings.") or name.startswith("encoder.layer.0.")
                    or name.startswith("pooler.")):
                raise ValueError(f"Unexpected tensor in reduced merged model: {name}")
            sst_param = sst_state[name]
            mnli_param = mnli_state[name]
            if target.shape != sst_param.shape or target.shape != mnli_param.shape:
                raise ValueError(f"Incompatible expert tensor: {name}")
            if torch.is_floating_point(target):
                merge_float_elements += target.numel()
                merge_float_tensors += 1
                target.copy_(MERGE_ALPHA_SST * sst_param + MERGE_BETA_MNLI * mnli_param)
            else:
                target.copy_(sst_param)

    del sst_state, mnli_state, sst_param, mnli_param, target
    merge_cost = {
        "method": "executed_weight_average_formula_v1",
        "scope": "one-time preparation: merged embeddings, encoder layer 1 and pooler only; excluded from inference",
        "alpha_sst": float(MERGE_ALPHA_SST), "beta_mnli": float(MERGE_BETA_MNLI),
        "floating_elements": int(merge_float_elements),
        "floating_tensors": int(merge_float_tensors),
        "flops_per_element": 3,
        "total_flops": int(3 * merge_float_elements),
        "convention": "two scalar multiplications and one addition per floating weight; copies/I/O excluded",
    }
    merged_backbone.eval()
    for param in merged_backbone.parameters():
        param.requires_grad = False
    assert len(merged_backbone.encoder.layer) == 1

    # ============================================================
    # CLASSIFIER HEAD
    # ============================================================

    class ClassifierHead(nn.Module):
        def __init__(self, input_dim=768, hidden_dim=256, num_classes=2, dropout_prob=0.1):
            super().__init__()

            self.classifier = nn.Sequential(
                nn.LayerNorm(input_dim),
                nn.Dropout(dropout_prob),
                nn.Linear(input_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout_prob),
                nn.Linear(hidden_dim, num_classes)
            )

        def forward(self, x):
            return self.classifier(x)

    ClassifierHead.__qualname__ = 'ClassifierHead'

    # ============================================================
    # INITIALIZE TASK-SPECIFIC CLASSIFIERS
    # ============================================================

    HIDDEN_SIZE = 768
    HEAD_HIDDEN_SIZE = 256
    DROPOUT_PROBABILITY = 0.1

    sst_head = ClassifierHead(
        input_dim=HIDDEN_SIZE,
        hidden_dim=HEAD_HIDDEN_SIZE,
        num_classes=2,
        dropout_prob=DROPOUT_PROBABILITY
    ).to(DEVICE)

    mnli_head = ClassifierHead(
        input_dim=HIDDEN_SIZE,
        hidden_dim=HEAD_HIDDEN_SIZE,
        num_classes=3,
        dropout_prob=DROPOUT_PROBABILITY
    ).to(DEVICE)

    # ============================================================
    # PRINT CHECK
    # ============================================================

    def count_trainable_params(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("\nModels loaded correctly.")
    print("SST backbone trainable params:   ", count_trainable_params(sst_backbone))
    print("MNLI backbone trainable params:  ", count_trainable_params(mnli_backbone))
    print("Merged backbone trainable params:", count_trainable_params(merged_backbone))
    print("SST head trainable params:       ", count_trainable_params(sst_head))
    print("MNLI head trainable params:      ", count_trainable_params(mnli_head))

    print("\nMerged backbone weights:")
    print("SST alpha :", MERGE_ALPHA_SST)
    print("MNLI beta :", MERGE_BETA_MNLI)
    print("Sum       :", MERGE_ALPHA_SST + MERGE_BETA_MNLI)

In [ ]:
def extract_layers():
    global NUM_LAYERS, merged_layers, mnli_layers, module, param, shared_embeddings, sst_layers
    # ============================================================
    # EXTRACT EMBEDDINGS AND ENCODER LAYERS
    # ============================================================

    # Shared embeddings
    shared_embeddings = merged_backbone.embeddings

    # Encoder layers
    sst_layers = list(sst_backbone.encoder.layer)
    mnli_layers = list(mnli_backbone.encoder.layer)
    merged_layers = list(merged_backbone.encoder.layer)

    # ============================================================
    # FREEZE EVERYTHING AGAIN FOR SAFETY
    # ============================================================

    for module in [shared_embeddings, *sst_layers, *mnli_layers, *merged_layers]:
        for param in module.parameters():
            param.requires_grad = False

    # ============================================================
    # CHECKS
    # ============================================================

    NUM_LAYERS = len(sst_layers)

    assert len(sst_layers) == len(mnli_layers), "Expert layer lists have different lengths"
    assert len(merged_layers) == 1, "Only merged encoder layer 1 must exist"
    assert NUM_LAYERS == 12, "Expected 12 BERT layers"

    print("Shared embeddings: merged_backbone.embeddings")
    print("Number of SST layers:   ", len(sst_layers))
    print("Number of MNLI layers:  ", len(mnli_layers))
    print("Number of merged layers:", len(merged_layers))

    print("Embedding trainable params:", count_trainable_params(shared_embeddings))
    print("SST layers trainable params:", sum(count_trainable_params(layer) for layer in sst_layers))
    print("MNLI layers trainable params:", sum(count_trainable_params(layer) for layer in mnli_layers))
    print("Merged layers trainable params:", sum(count_trainable_params(layer) for layer in merged_layers))

In [ ]:
def initialize_router():
    global CHOICE_IDENTITY, CHOICE_MERGED, CHOICE_MNLI, CHOICE_SST, DEPTH_EMBEDDING_SIZE, F, NUM_CHOICES, NUM_LAYERS
    global ROUTER_HIDDEN_SIZES, SharedMLPRouter, nn, router, torch
    # ============================================================
    # SHARED MLP ROUTER
    # One router reused at every depth: CLS + depth -> 64 -> 32 -> 4
    # ============================================================

    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    # ============================================================
    # ROUTER PARAMETERS
    # ============================================================

    NUM_CHOICES = 4
    CHOICE_SST = 0
    CHOICE_MNLI = 1
    CHOICE_MERGED = 2
    CHOICE_IDENTITY = 3

    ROUTER_HIDDEN_SIZES = (64, 32)
    DEPTH_EMBEDDING_SIZE = 32
    NUM_LAYERS = 12

    # ============================================================
    # ROUTER MODULE
    # ============================================================

    class SharedMLPRouter(nn.Module):
        def __init__(
            self,
            hidden_size=768,
            router_hidden_sizes=(64, 32),
            depth_embedding_size=32,
            num_layers=12,
            num_choices=4,
            merged_mode="auto"
        ):
            super().__init__()

            self.hidden_size = hidden_size
            self.num_layers = num_layers
            self.num_choices = num_choices
            if num_choices != 4:
                raise ValueError("The export schema requires SST, MNLI, MERGED, IDENTITY")
            self.merged_mode = merged_mode
            self.available_choices_by_layer = [
                merged_choices_for_layer(merged_mode, depth) for depth in range(num_layers)
            ]
            choice_names = ("SST", "MNLI", "MERGED", "IDENTITY")
            unavailable = torch.tensor([
                [name not in available for name in choice_names]
                for available in self.available_choices_by_layer
            ], dtype=torch.bool)
            self.register_buffer("unavailable_choices", unavailable)

            self.depth_embedding = nn.Embedding(
                num_embeddings=num_layers,
                embedding_dim=depth_embedding_size
            )

            router_input_size = hidden_size + depth_embedding_size

            # These same MLP weights are used for every layer depth.
            self.router_mlp = nn.Sequential(
                nn.Linear(router_input_size, router_hidden_sizes[0]),
                nn.GELU(),
                nn.Linear(router_hidden_sizes[0], router_hidden_sizes[1]),
                nn.GELU(),
                nn.Linear(router_hidden_sizes[1], num_choices)
            )

        def forward_step(
            self,
            cls_hidden,
            layer_idx,
            tau=1.0
        ):
            batch_size = cls_hidden.size(0)
            device = cls_hidden.device

            depth_idx = torch.full(
                size=(batch_size,),
                fill_value=layer_idx,
                dtype=torch.long,
                device=device
            )

            depth_emb = self.depth_embedding(depth_idx)
            router_input = torch.cat([cls_hidden, depth_emb], dim=1)

            logits = self.router_mlp(router_input)
            # Mask BEFORE softmax/Gumbel/argmax: forbidden choice has zero probability
            # and receives no gradient through these decisions.
            logits = logits.masked_fill(self.unavailable_choices[layer_idx], float("-inf"))
            soft_probs = F.softmax(logits, dim=1)

            if self.training:
                hard_choice = F.gumbel_softmax(
                    logits,
                    tau=tau,
                    hard=True,
                    dim=1
                )
            else:
                choice_idx = torch.argmax(logits, dim=1)
                hard_choice = F.one_hot(choice_idx, num_classes=self.num_choices).float()

            return logits, soft_probs, hard_choice

    SharedMLPRouter.__qualname__ = 'SharedMLPRouter'

    # ============================================================
    # INITIALIZE ONE SHARED ROUTER
    # ============================================================

    router = SharedMLPRouter(
        hidden_size=HIDDEN_SIZE,
        router_hidden_sizes=ROUTER_HIDDEN_SIZES,
        depth_embedding_size=DEPTH_EMBEDDING_SIZE,
        num_layers=NUM_LAYERS,
        num_choices=NUM_CHOICES,
        merged_mode=MERGED_MODE
    ).to(DEVICE)

    # ============================================================
    # PRINT CHECK
    # ============================================================

    print("Shared MLP router initialized (same weights at every depth).")
    print("Router trainable params:", count_trainable_params(router))
    print("Merged mode:", router.merged_mode, "| choices:", router.available_choices_by_layer)


In [ ]:
def initialize_routed_model():
    global RoutedBertModel, extract_layer_hidden_states, nn, param, routed_model, torch, validate_merged_layout
    # ============================================================
    # ROUTED BERT MODEL
    # Training evaluates available branches only; efficient inference; merged pooler
    # Fixed layer output extraction
    # ============================================================

    import torch
    import torch.nn as nn

    # ============================================================
    # SAFE LAYER OUTPUT EXTRACTION
    # ============================================================

    def extract_layer_hidden_states(layer_output):
        """
        Handles both cases:
        - layer_output is already a tensor: [batch, seq_len, hidden]
        - layer_output is a tuple/list: first element is hidden states
        """

        if torch.is_tensor(layer_output):
            return layer_output

        if isinstance(layer_output, (tuple, list)):
            return layer_output[0]

        if hasattr(layer_output, "last_hidden_state"):
            return layer_output.last_hidden_state

        raise TypeError(f"Unexpected layer output type: {type(layer_output)}")


    class RoutedBertModel(nn.Module):
        def __init__(
            self,
            shared_embeddings,
            sst_layers,
            mnli_layers,
            merged_layers,
            router,
            sst_head,
            mnli_head,
            reference_backbone,
            num_choices=4
        ):
            super().__init__()

            self.shared_embeddings = shared_embeddings

            self.sst_layers = nn.ModuleList(sst_layers)
            self.mnli_layers = nn.ModuleList(mnli_layers)
            self.merged_layers = nn.ModuleList(merged_layers)

            self.pooler = reference_backbone.pooler

            self.router = router
            self.sst_head = sst_head
            self.mnli_head = mnli_head

            self.reference_backbone = reference_backbone
            self.num_choices = num_choices
            self.num_layers = len(sst_layers)
            if len(merged_layers) != 1 or len(reference_backbone.encoder.layer) != 1:
                raise ValueError("Merged must physically contain only encoder layer 1")

        def apply_wasteful_layer(
            self,
            hidden_states,
            extended_attention_mask,
            layer_idx,
            hard_choice
        ):
            batch_size = hidden_states.size(0)

            # A forced first layer executes only the merged encoder branch.
            if layer_idx == 0 and self.router.merged_mode == "impose":
                return extract_layer_hidden_states(
                    self.merged_layers[0](hidden_states, attention_mask=extended_attention_mask)
                )

            sst_output = extract_layer_hidden_states(
                self.sst_layers[layer_idx](
                    hidden_states,
                    attention_mask=extended_attention_mask
                )
            )

            mnli_output = extract_layer_hidden_states(
                self.mnli_layers[layer_idx](
                    hidden_states,
                    attention_mask=extended_attention_mask
                )
            )

            choice_sst = hard_choice[:, CHOICE_SST].view(batch_size, 1, 1)
            choice_mnli = hard_choice[:, CHOICE_MNLI].view(batch_size, 1, 1)
            choice_identity = hard_choice[:, CHOICE_IDENTITY].view(batch_size, 1, 1)
            combined = choice_sst * sst_output + choice_mnli * mnli_output
            if layer_idx == 0 and self.router.merged_mode == "auto":
                merged_output = extract_layer_hidden_states(
                    self.merged_layers[0](hidden_states, attention_mask=extended_attention_mask)
                )
                choice_merged = hard_choice[:, CHOICE_MERGED].view(batch_size, 1, 1)
                combined = combined + choice_merged * merged_output
            hidden_states = combined + choice_identity * hidden_states

            return hidden_states

        def apply_efficient_layer(
            self,
            hidden_states,
            extended_attention_mask,
            layer_idx,
            hard_choice
        ):
            choice_idx = torch.argmax(hard_choice, dim=1)

            new_hidden_states = hidden_states.clone()

            for choice in range(self.num_choices):

                mask = choice_idx == choice

                if not mask.any():
                    continue

                if ("SST", "MNLI", "MERGED", "IDENTITY")[choice] not in self.router.available_choices_by_layer[layer_idx]:
                    raise RuntimeError("Attempted to execute an unavailable encoder choice")
                if choice == CHOICE_IDENTITY:
                    continue

                selected_hidden_states = hidden_states[mask]
                selected_attention_mask = extended_attention_mask[mask]

                if choice == CHOICE_SST:
                    selected_output = extract_layer_hidden_states(
                        self.sst_layers[layer_idx](
                            selected_hidden_states,
                            attention_mask=selected_attention_mask
                        )
                    )

                elif choice == CHOICE_MNLI:
                    selected_output = extract_layer_hidden_states(
                        self.mnli_layers[layer_idx](
                            selected_hidden_states,
                            attention_mask=selected_attention_mask
                        )
                    )

                elif choice == CHOICE_MERGED:
                    if layer_idx != 0:
                        raise RuntimeError("MERGED is unavailable after encoder layer 1")
                    selected_output = extract_layer_hidden_states(
                        self.merged_layers[0](
                            selected_hidden_states,
                            attention_mask=selected_attention_mask
                        )
                    )

                new_hidden_states[mask] = selected_output

            return new_hidden_states

        def forward(
            self,
            input_ids,
            attention_mask,
            token_type_ids=None,
            task="sst",
            tau=1.0,
            return_routing=False,
            efficient_inference=False
        ):
            batch_size = input_ids.size(0)
            device = input_ids.device

            if token_type_ids is None:
                token_type_ids = torch.zeros_like(input_ids)

            hidden_states = self.shared_embeddings(
                input_ids=input_ids,
                token_type_ids=token_type_ids
            )

            input_shape = input_ids.size()

            extended_attention_mask = self.reference_backbone.get_extended_attention_mask(
                attention_mask,
                input_shape,
                dtype=hidden_states.dtype
            )

            cumulative_counts = torch.zeros(batch_size, self.num_choices, device=device)

            routing_logits = []
            routing_probs = []
            routing_choices = []

            use_efficient_inference = efficient_inference and not self.training

            for layer_idx in range(self.num_layers):

                cls_hidden = hidden_states[:, 0, :]

                router_logits, soft_probs, hard_choice = self.router.forward_step(
                    cls_hidden=cls_hidden,
                    layer_idx=layer_idx,
                    tau=tau
                )

                if use_efficient_inference:
                    hidden_states = self.apply_efficient_layer(
                        hidden_states=hidden_states,
                        extended_attention_mask=extended_attention_mask,
                        layer_idx=layer_idx,
                        hard_choice=hard_choice
                    )
                else:
                    hidden_states = self.apply_wasteful_layer(
                        hidden_states=hidden_states,
                        extended_attention_mask=extended_attention_mask,
                        layer_idx=layer_idx,
                        hard_choice=hard_choice
                    )

                cumulative_counts = cumulative_counts + hard_choice

                routing_logits.append(router_logits)
                routing_probs.append(soft_probs)
                routing_choices.append(hard_choice)

            if self.pooler is not None:
                cls_final = self.pooler(hidden_states)
            else:
                cls_final = hidden_states[:, 0, :]

            if task == "sst":
                logits = self.sst_head(cls_final)
            elif task == "mnli":
                logits = self.mnli_head(cls_final)
            else:
                raise ValueError("task must be 'sst' or 'mnli'")

            if return_routing:
                return {
                    "logits": logits,
                    "routing_logits": torch.stack(routing_logits, dim=1),
                    "routing_probs": torch.stack(routing_probs, dim=1),
                    "routing_choices": torch.stack(routing_choices, dim=1),
                    "cumulative_counts": cumulative_counts
                }

            return logits

    RoutedBertModel.__qualname__ = 'RoutedBertModel'


    # ============================================================
    # INITIALIZE ROUTED MODEL
    # ============================================================

    routed_model = RoutedBertModel(
        shared_embeddings=shared_embeddings,
        sst_layers=sst_layers,
        mnli_layers=mnli_layers,
        merged_layers=merged_layers,
        router=router,
        sst_head=sst_head,
        mnli_head=mnli_head,
        reference_backbone=merged_backbone,
        num_choices=NUM_CHOICES
    ).to(DEVICE)

    # ============================================================
    # FREEZE POOLER
    # ============================================================

    if routed_model.pooler is not None:
        for param in routed_model.pooler.parameters():
            param.requires_grad = False

    # ============================================================
    # PRINT CHECK
    # ============================================================

    print("Routed model initialized.")
    print("Trainable params:", count_trainable_params(routed_model))
    print("Frozen embeddings params:", count_trainable_params(shared_embeddings))
    print("Frozen pooler params:", count_trainable_params(routed_model.pooler) if routed_model.pooler is not None else 0)
    print("Frozen SST layers params:", sum(count_trainable_params(layer) for layer in sst_layers))
    print("Frozen MNLI layers params:", sum(count_trainable_params(layer) for layer in mnli_layers))
    print("Frozen merged layers params:", sum(count_trainable_params(layer) for layer in merged_layers))


    def validate_merged_layout(model):
        """Structural check outside timed/counted passes, also after checkpoint reload."""
        if model.num_layers != 12 or len(model.sst_layers) != 12 or len(model.mnli_layers) != 12:
            raise ValueError("Expected twelve routing positions and two full expert stacks")
        if len(model.merged_layers) != 1 or len(model.reference_backbone.encoder.layer) != 1:
            raise ValueError("Checkpoint contains unexpected merged encoder layers")
        if model.merged_layers[0] is not model.reference_backbone.encoder.layer[0]:
            raise ValueError("Merged first layer is duplicated instead of shared")
        if model.shared_embeddings is not model.reference_backbone.embeddings:
            raise ValueError("Shared merged embeddings do not match the reference")
        if model.pooler is not model.reference_backbone.pooler:
            raise ValueError("Shared merged pooler does not match the reference")
        expected_choices = [
            merged_choices_for_layer(model.router.merged_mode, depth) for depth in range(12)
        ]
        if model.router.available_choices_by_layer != expected_choices:
            raise ValueError("Router choices do not match its merged mode")
        expected_mask = [
            [name not in available for name in ("SST", "MNLI", "MERGED", "IDENTITY")]
            for available in expected_choices
        ]
        if model.router.unavailable_choices.cpu().tolist() != expected_mask:
            raise ValueError("Router availability mask does not match the reduced model")


    validate_merged_layout(routed_model)


In [ ]:
def initialize_optimizer_and_counters():
    global AdamW, EARLY_STOPPING_MIN_DELTA, EARLY_STOPPING_PATIENCE, EarlyStopping, FlopCounterMode, GRADIENT_CLIPPING, HEAD_LEARNING_RATE, HEAD_WEIGHT_DECAY
    global InferenceFlopCounter, LEARNING_RATE, LR_FACTOR, LR_PATIENCE, MAX_GRAD_NORM, MIN_HEAD_LR, MIN_LR, MIN_ROUTER_LR
    global ROUTER_LEARNING_RATE, ROUTER_WEIGHT_DECAY, ReduceLROnPlateau, RoutedFlopCounter, TrainingFlopCounter, WEIGHT_DECAY, add_inference_cost, add_training_cost
    global count_total_params, count_trainable_params, defaultdict, early_stopping, finish_inference_cost, finish_multitask_training_cost, finish_routed_training_cost, finish_training_cost
    global freeze_module, group, group_name, group_params, isfinite, json_config_value, merge_training_cost, mnli_head_params
    global new_inference_cost, new_training_cost, optimizer, param, print_module_status, prod, router_params, scheduler
    global set_routed_model_eval_mode, set_routed_model_train_mode, sst_head_params, total_optimizer_params, trainable_params_model, transformers_version, tree_leaves, unfreeze_module
    # ============================================================
    # OPTIMIZER, SCHEDULER, FREEZING, AND ANTI-OVERFITTING
    # Different LR for router and heads
    # ============================================================

    from torch.optim import AdamW
    from torch.optim.lr_scheduler import ReduceLROnPlateau

    # ============================================================
    # TRAINING PARAMETERS
    # ============================================================

    ROUTER_LEARNING_RATE = 1e-4
    HEAD_LEARNING_RATE = 1e-3

    ROUTER_WEIGHT_DECAY = 0.01
    HEAD_WEIGHT_DECAY = 0.01

    LR_FACTOR = 0.5
    LR_PATIENCE = 2

    MIN_ROUTER_LR = 1e-6
    MIN_HEAD_LR = 1e-6

    EARLY_STOPPING_PATIENCE = 4
    EARLY_STOPPING_MIN_DELTA = 1e-4

    GRADIENT_CLIPPING = True
    MAX_GRAD_NORM = 1.0

    # Compatibility names for saving blocks
    LEARNING_RATE = HEAD_LEARNING_RATE
    WEIGHT_DECAY = HEAD_WEIGHT_DECAY
    MIN_LR = MIN_HEAD_LR

    # ============================================================
    # FREEZE / UNFREEZE HELPERS
    # ============================================================

    def freeze_module(module):
        if module is not None:
            for param in module.parameters():
                param.requires_grad = False


    def unfreeze_module(module):
        if module is not None:
            for param in module.parameters():
                param.requires_grad = True


    # ============================================================
    # FREEZE / UNFREEZE CORRECT MODULES
    # ============================================================

    freeze_module(routed_model.shared_embeddings)
    freeze_module(routed_model.pooler)
    freeze_module(routed_model.sst_layers)
    freeze_module(routed_model.mnli_layers)
    freeze_module(routed_model.merged_layers)

    unfreeze_module(routed_model.router)
    unfreeze_module(routed_model.sst_head)
    unfreeze_module(routed_model.mnli_head)

    # ============================================================
    # MODE MANAGEMENT
    # ============================================================

    routed_model.shared_embeddings.eval()

    if routed_model.pooler is not None:
        routed_model.pooler.eval()

    routed_model.sst_layers.eval()
    routed_model.mnli_layers.eval()
    routed_model.merged_layers.eval()

    routed_model.router.train()
    routed_model.sst_head.train()
    routed_model.mnli_head.train()


    def set_routed_model_train_mode(model):
        model.train()

        model.shared_embeddings.eval()

        if model.pooler is not None:
            model.pooler.eval()

        model.sst_layers.eval()
        model.mnli_layers.eval()
        model.merged_layers.eval()

        model.router.train()
        model.sst_head.train()
        model.mnli_head.train()


    def set_routed_model_eval_mode(model):
        model.eval()


    # ============================================================
    # PARAMETER COUNTERS
    # ============================================================

    def count_total_params(module):
        if module is None:
            return 0
        return sum(p.numel() for p in module.parameters())


    def count_trainable_params(module):
        if module is None:
            return 0
        return sum(p.numel() for p in module.parameters() if p.requires_grad)


    def print_module_status(name, module):
        total = count_total_params(module)
        trainable = count_trainable_params(module)
        frozen = total - trainable

        status = "TRAINABLE" if trainable > 0 else "FROZEN"

        print(
            f"{name:<25} | {status:<10} | "
            f"total: {total:<12} | trainable: {trainable:<12} | frozen: {frozen:<12}"
        )


    # ============================================================
    # OPTIMIZER PARAMETER GROUPS
    # ============================================================

    router_params = [
        param for param in routed_model.router.parameters()
        if param.requires_grad
    ]

    sst_head_params = [
        param for param in routed_model.sst_head.parameters()
        if param.requires_grad
    ]

    mnli_head_params = [
        param for param in routed_model.mnli_head.parameters()
        if param.requires_grad
    ]

    optimizer = AdamW(
        [
            {
                "params": router_params,
                "lr": ROUTER_LEARNING_RATE,
                "weight_decay": ROUTER_WEIGHT_DECAY,
                "name": "router"
            },
            {
                "params": sst_head_params,
                "lr": HEAD_LEARNING_RATE,
                "weight_decay": HEAD_WEIGHT_DECAY,
                "name": "sst_head"
            },
            {
                "params": mnli_head_params,
                "lr": HEAD_LEARNING_RATE,
                "weight_decay": HEAD_WEIGHT_DECAY,
                "name": "mnli_head"
            }
        ]
    )

    # ============================================================
    # OPTIMIZER PARAMETER CHECK
    # ============================================================

    print("=" * 95)
    print("OPTIMIZER PARAMETER GROUPS")
    print("=" * 95)

    total_optimizer_params = 0

    for group in optimizer.param_groups:
        group_name = group.get("name", "unnamed")
        group_params = sum(param.numel() for param in group["params"])
        total_optimizer_params += group_params

        print(
            f"{group_name:<20} | "
            f"params: {group_params:<12} | "
            f"lr: {group['lr']:<10} | "
            f"weight_decay: {group['weight_decay']}"
        )

    print("-" * 95)
    print("Total optimizer params:", total_optimizer_params)

    trainable_params_model = count_trainable_params(routed_model)
    print("Trainable params model:", trainable_params_model)

    assert total_optimizer_params == trainable_params_model, (
        "Optimizer params do not match trainable model params."
    )

    # ============================================================
    # SCHEDULER
    # ============================================================

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        min_lr=[MIN_ROUTER_LR, MIN_HEAD_LR, MIN_HEAD_LR]
    )

    # ============================================================
    # EARLY STOPPING
    # ============================================================

    class EarlyStopping:
        def __init__(self, patience=4, min_delta=1e-4):
            self.patience = patience
            self.min_delta = min_delta
            self.best_score = None
            self.counter = 0
            self.should_stop = False

        def step(self, score):
            if self.best_score is None:
                self.best_score = score
                return True

            if score > self.best_score + self.min_delta:
                self.best_score = score
                self.counter = 0
                return True

            self.counter += 1

            if self.counter >= self.patience:
                self.should_stop = True

            return False

    EarlyStopping.__qualname__ = 'EarlyStopping'


    early_stopping = EarlyStopping(
        patience=EARLY_STOPPING_PATIENCE,
        min_delta=EARLY_STOPPING_MIN_DELTA
    )

    # ============================================================
    # PRINT STATUS
    # ============================================================

    print("\n" + "=" * 95)
    print("FREEZE / UNFREEZE STATUS")
    print("=" * 95)

    print_module_status("Shared embeddings", routed_model.shared_embeddings)
    print_module_status("Merged pooler", routed_model.pooler)
    print_module_status("SST layers", routed_model.sst_layers)
    print_module_status("MNLI layers", routed_model.mnli_layers)
    print_module_status("Merged layers", routed_model.merged_layers)
    print_module_status("Router", routed_model.router)
    print_module_status("SST head", routed_model.sst_head)
    print_module_status("MNLI head", routed_model.mnli_head)

    print("-" * 95)
    print_module_status("Full routed model", routed_model)

    print("=" * 95)
    print("OPTIMIZER / REGULARIZATION")
    print("=" * 95)
    print("Router learning rate:    ", ROUTER_LEARNING_RATE)
    print("Head learning rate:      ", HEAD_LEARNING_RATE)
    print("Router weight decay:     ", ROUTER_WEIGHT_DECAY)
    print("Head weight decay:       ", HEAD_WEIGHT_DECAY)
    print("Scheduler:               ReduceLROnPlateau")
    print("LR factor:               ", LR_FACTOR)
    print("LR patience:             ", LR_PATIENCE)
    print("Min router LR:           ", MIN_ROUTER_LR)
    print("Min head LR:             ", MIN_HEAD_LR)
    print("Early stopping patience: ", EARLY_STOPPING_PATIENCE)
    print("Early stopping delta:    ", EARLY_STOPPING_MIN_DELTA)
    print("Gradient clipping:       ", GRADIENT_CLIPPING)
    print("Max grad norm:           ", MAX_GRAD_NORM)
    # ============================================================
    # COMPUTATIONAL COST: INFERENCE AND ROUTER TRAINING
    # ============================================================
    from math import prod, isfinite
    from collections import defaultdict
    from torch.utils.flop_counter import FlopCounterMode
    from torch.utils._pytree import tree_leaves
    from transformers import __version__ as transformers_version


    class InferenceFlopCounter(FlopCounterMode):
        """Extend PyTorch's matrix counts and expose uncovered floating-point ops."""

        # Indexing, copies, shape changes, allocation and comparisons have no
        # floating-point arithmetic under this convention. Dropout is off in eval.
        zero_flop_ops = set("""
            alias detach view _unsafe_view reshape t transpose permute slice select
            squeeze unsqueeze expand expand_as as_strided contiguous clone copy_
            _to_copy to empty empty_like empty_strided new_empty new_empty_strided
            zeros zeros_like new_zeros ones ones_like full full_like new_full fill_
            zero_ embedding index index_select gather cat stack split split_with_sizes
            unbind masked_fill masked_fill_ where eq ne lt le gt ge argmax argmin
            max min amax amin any all isfinite isnan isinf scalar_tensor
            _local_scalar_dense lift_fresh lift_fresh_copy resolve_conj dropout
            native_dropout constant_pad_nd
        """.split())

        def __init__(self):
            if not hasattr(FlopCounterMode, "_count_flops"):
                raise RuntimeError("This FLOPs counter requires PyTorch with FlopCounterMode._count_flops support.")
            super().__init__(display=False)
            self.uncovered_ops = set()

        def _count_flops(self, packet, out, args, kwargs):
            name = str(packet).split(".")[-1]
            outputs = [x for x in tree_leaves(out)
                       if isinstance(x, torch.Tensor) and x.is_floating_point()]
            if not outputs:
                return out
            n = outputs[0].numel()
            x = args[0] if args else None

            if name in {"addmm", "baddbmm"}:
                # PyTorch's default matrix formula omits the bias addition.
                alpha, beta = kwargs.get("alpha", 1), kwargs.get("beta", 1)
                flops = 2 * n * args[1].shape[-1]
                flops += n * (int(alpha != 1) + int(beta != 0) + int(beta not in (0, 1)))
            elif name in {"add", "add_", "sub", "sub_", "rsub"}:
                flops = n * (1 + int(kwargs.get("alpha", 1) != 1))
            elif name in {"mul", "mul_", "div", "div_", "true_divide", "neg",
                          "exp", "log", "sqrt", "rsqrt", "reciprocal", "tanh", "erf", "pow"}:
                flops = n
            elif name == "gelu":
                flops = n * (9 if kwargs.get("approximate", "none") == "tanh" else 5)
            elif name in {"native_layer_norm", "layer_norm"}:
                width = prod(args[1])
                rows = x.numel() // width
                weight = args[2] if len(args) > 2 else kwargs.get("weight")
                bias = args[3] if len(args) > 3 else kwargs.get("bias")
                flops = 5 * x.numel() + 2 * rows
                flops += x.numel() * (int(weight is not None) + int(bias is not None))
            elif name in {"_softmax", "softmax"}:
                dim = args[1] if len(args) > 1 else kwargs["dim"]
                flops = 4 * x.numel() - x.numel() // x.shape[dim]
            elif name in {"sum", "mean"}:
                flops = x.numel() - n + (n if name == "mean" else 0)
            elif name in {
                "scaled_dot_product_attention", "_scaled_dot_product_attention_math",
                "_scaled_dot_product_flash_attention", "_scaled_dot_product_flash_attention_for_cpu",
                "_scaled_dot_product_efficient_attention", "_scaled_dot_product_cudnn_attention"
            }:
                # Dense, non-causal BERT attention: QK + scaling/mask/softmax + AV.
                # Applies to the operator as a whole; no extra forward is performed.
                q, k, v = args[:3]
                rows = prod(q.shape[:-1])
                scores = rows * k.shape[-2]
                mask = kwargs.get("attn_mask", kwargs.get("attn_bias"))
                if mask is None and len(args) > 3 and isinstance(args[3], torch.Tensor):
                    mask = args[3]
                flops = 2 * scores * (q.shape[-1] + v.shape[-1])
                flops += 5 * scores - rows
                if isinstance(mask, torch.Tensor) and mask.is_floating_point():
                    flops += scores
            elif name in self.zero_flop_ops:
                return out
            elif packet in self.flop_registry:
                # Other fused attention variants may count only their matmuls.
                if "attention" in name:
                    self.uncovered_ops.add(str(packet) + " (attention overhead not covered)")
                return super()._count_flops(packet, out, args, kwargs)
            else:
                self.uncovered_ops.add(str(packet))
                return out

            self.flop_counts["Global"][packet] += int(flops)
            return out

    InferenceFlopCounter.__qualname__ = 'InferenceFlopCounter'


    def new_inference_cost():
        return {
            "components": defaultdict(int), "operators": defaultdict(int),
            "uncovered": set(), "num_examples": 0,
            "sequence_lengths": set(), "batch_sizes": set(),
        }


    def add_inference_cost(cost, counter, component, input_ids=None):
        cost["components"][component] += int(counter.get_total_flops())
        for op, value in counter.get_flop_counts().get("Global", {}).items():
            cost["operators"][str(op)] += int(value)
        cost["uncovered"].update(counter.uncovered_ops)
        # Count examples only on the backbone pass, not again on cached features.
        if input_ids is not None:
            cost["num_examples"] += int(input_ids.shape[0])
            cost["sequence_lengths"].add(int(input_ids.shape[1]))
            cost["batch_sizes"].add(int(input_ids.shape[0]))


    def finish_inference_cost(cost, num_samples):
        if num_samples <= 0 or cost["num_examples"] != num_samples:
            raise ValueError("FLOPs: backbone and classifier must cover the same test examples.")
        counted = int(sum(cost["components"].values()))
        complete = not cost["uncovered"] and counted > 0
        return {
            "method": "pytorch_executed_forward_flops_v1",
            "scope": "test inference: shared embeddings, router, selected BERT layers, pooler and task head; inputs to logits",
            "total_flops": counted if complete else None,
            "flops_per_example": float(counted / num_samples) if complete else None,
            "num_examples": int(num_samples),
            "coverage": {
                "status": "complete_for_convention" if complete else "incomplete",
                "uncovered_operators": sorted(cost["uncovered"]),
                "counted_flops": counted,
                "counted_flops_by_component": dict(cost["components"]),
                "counted_flops_by_operator": dict(sorted(cost["operators"].items())),
            },
            "convention": {
                "multiply_accumulate_flops": 2,
                "elementwise_arithmetic_or_special_function_flops": 1,
                "gelu_flops_per_element": {"exact": 5, "tanh_approximation": 9},
                "layer_norm": "5*N + 2*rows, plus N each for affine weight and bias",
                "softmax": "4*N - rows; comparisons excluded",
                "router": "shared MLP matmuls, biases, two GELU activations and routing arithmetic; depth embedding lookup has zero arithmetic FLOPs",
                "attention": "dense QK and AV matmuls plus scaling, additive mask and softmax",
                "excluded": "training/backward, loss/metrics, preprocessing, I/O, copies, indexing, comparisons",
                "interpretation": "formula-based FLOPs of executed operators; not exact GPU kernel instructions",
            },
            "input": {
                "max_len": int(MAX_LEN), "batch_size": int(BATCH_SIZE),
                "observed_sequence_lengths": sorted(cost["sequence_lengths"]),
                "observed_batch_sizes": sorted(cost["batch_sizes"]),
                "padding": "longest_in_split", "truncation": True,
                "tokenizer": str(TOKENIZER_NAME),
                "dtype": str(next(routed_model.parameters()).dtype),
            },
            "torch_version": str(torch.__version__),
            "transformers_version": transformers_version,
        }



    # ============================================================
    # TRAINING FLOPS: extend the same convention to the actual train step
    # ============================================================
    class TrainingFlopCounter(InferenceFlopCounter):
        """Count training tensor arithmetic without another forward/backward pass."""

        def __init__(self, count_attention_saved_stats=False, embedding_padding_counts=None):
            super().__init__()
            self.count_attention_saved_stats = count_attention_saved_stats
            # Counts come from the existing CPU input batch, without reading GPU tensors.
            self.embedding_padding_counts = embedding_padding_counts or {}

        zero_flop_ops = InferenceFlopCounter.zero_flop_ops | {
            "select_backward", "slice_backward", "diagonal_backward", "_reshape_alias",
            "as_strided_", "constant_pad_nd", "detach_", "unbind_backward",
        }


        def _count_flops(self, packet, out, args, kwargs):
            name = str(packet).split(".")[-1]
            outputs = [x for x in tree_leaves(out)
                       if isinstance(x, torch.Tensor) and x.is_floating_point()]
            n = outputs[0].numel() if outputs else 0


            # BERT pooler and embedding gradients, in addition to the head operators.
            if name == "tanh_backward":
                self.flop_counts["Global"][packet] += 3 * args[0].numel()
                return out
            if name in {"embedding_dense_backward", "embedding_backward"}:
                grad, indices, num_weights, padding_idx, scale_by_frequency = args[:5]
                sparse = args[5] if len(args) > 5 else False
                if scale_by_frequency or sparse:
                    self.uncovered_ops.add(str(packet) + " (frequency scaling/sparse gradient not covered)")
                    return out
                valid_rows = indices.numel()
                if padding_idx >= 0:
                    key = (int(num_weights), int(padding_idx), int(indices.numel()))
                    if key not in self.embedding_padding_counts:
                        self.uncovered_ops.add(str(packet) + " (padding count unavailable)")
                        return out
                    valid_rows -= self.embedding_padding_counts[key]
                # One gradient accumulation per embedding element and non-padding occurrence.
                self.flop_counts["Global"][packet] += int(valid_rows * grad.shape[-1])
                return out
            if name in {"index_add", "index_add_", "scatter_add", "scatter_add_"}:
                self.flop_counts["Global"][packet] += args[3].numel() * (1 + int(kwargs.get("alpha", 1) != 1))
                return out

            # Fused BERT attention: dense, non-causal, equal query/key head counts.
            forward_attention = {
                "scaled_dot_product_attention", "_scaled_dot_product_attention_math",
                "_scaled_dot_product_flash_attention", "_scaled_dot_product_flash_attention_for_cpu",
                "_scaled_dot_product_efficient_attention", "_scaled_dot_product_cudnn_attention",
            }
            backward_attention = {
                "_scaled_dot_product_efficient_attention_backward",
                "_scaled_dot_product_flash_attention_backward",
                "_scaled_dot_product_flash_attention_for_cpu_backward",
                "_scaled_dot_product_cudnn_attention_backward",
            }
            if name in forward_attention or name in backward_attention:
                def argument(index, key, default=None):
                    return args[index] if len(args) > index else kwargs.get(key, default)

                is_backward = name in backward_attention
                q, k, v = args[1:4] if is_backward else args[:3]
                if q.ndim != 4 or k.ndim != 4 or v.ndim != 4 or q.shape[:2] != k.shape[:2] or k.shape[:2] != v.shape[:2]:
                    self.uncovered_ops.add(str(packet) + " (non-standard attention shape)")
                    return out
                mask = None
                save_stats = False
                if not is_backward:
                    if name in {"_scaled_dot_product_efficient_attention", "_scaled_dot_product_cudnn_attention"}:
                        mask = argument(3, "attn_bias")
                        save_stats = bool(argument(4, "compute_log_sumexp", False))
                        dropout = argument(5, "dropout_p", 0.0)
                        causal = argument(6, "is_causal", False)
                    elif name in {"_scaled_dot_product_flash_attention", "_scaled_dot_product_flash_attention_for_cpu"}:
                        mask = kwargs.get("attn_mask")
                        dropout = argument(3, "dropout_p", 0.0)
                        causal = argument(4, "is_causal", False)
                        save_stats = self.count_attention_saved_stats
                    else:
                        mask = argument(3, "attn_mask")
                        dropout = argument(4, "dropout_p", 0.0)
                        causal = argument(5, "is_causal", False)
                elif name == "_scaled_dot_product_efficient_attention_backward":
                    mask = argument(4, "attn_bias")
                    dropout = argument(9, "dropout_p", 0.0)
                    grad_mask = argument(10, "grad_input_mask")
                    causal = argument(11, "is_causal", False)
                    if grad_mask is None or not all(grad_mask[:3]) or grad_mask[3]:
                        self.uncovered_ops.add(str(packet) + " (partial QKV or bias gradient not covered)")
                        return out
                elif name == "_scaled_dot_product_cudnn_attention_backward":
                    mask = argument(8, "attn_bias")
                    dropout = argument(13, "dropout_p", 0.0)
                    causal = argument(14, "is_causal", False)
                elif name == "_scaled_dot_product_flash_attention_for_cpu_backward":
                    mask = kwargs.get("attn_mask")
                    dropout = argument(6, "dropout_p", 0.0)
                    causal = argument(7, "is_causal", False)
                else:
                    dropout = argument(10, "dropout_p", 0.0)
                    causal = argument(11, "is_causal", False)
                if causal or dropout < 0 or dropout >= 1:
                    self.uncovered_ops.add(str(packet) + " (causal attention or unsupported dropout)")
                    return out

                rows = prod(q.shape[:-1])
                scores = rows * k.shape[-2]
                mask_additions = scores if isinstance(mask, torch.Tensor) and mask.is_floating_point() else 0
                if not is_backward:
                    # Same inference formula, plus active dropout and saved log-sum-exp.
                    flops = 2 * scores * (q.shape[-1] + v.shape[-1])
                    flops += 5 * scores - rows + mask_additions
                    flops += 2 * scores if dropout > 0 else 0
                    flops += 2 * rows if save_stats and self.count_attention_saved_stats else 0
                else:
                    # Fused backward recomputes QK and reconstructs P from saved logsumexp.
                    # Five matmuls: QK, dP, dV, dQ, dK. delta=sum(dO*O) per row.
                    flops = 2 * scores * (3 * q.shape[-1] + 2 * v.shape[-1])
                    flops += 6 * scores + mask_additions + (2 * args[0].numel() - rows)
                    # Apply the reused dropout mask both to reconstructed P and to dP.
                    flops += 4 * scores if dropout > 0 else 0
                self.flop_counts["Global"][packet] += int(flops)
                return out

            # RNG generation is excluded; applying/scaling the dropout mask is counted.
            if name in {"bernoulli", "bernoulli_", "uniform_", "normal_"}:
                return out
            if name in {"dropout", "native_dropout"}:
                probability = args[1] if len(args) > 1 else kwargs["p"]
                training = args[2] if len(args) > 2 else kwargs.get("train", True)
                flops = 2 * args[0].numel() if training and 0 < probability < 1 else 0
            elif name == "native_dropout_backward":
                flops = 2 * args[0].numel()
            elif name == "gelu_backward":
                # Exact: g * (Phi(x) + x * phi(x)); scalar constants precomputed.
                approximation = kwargs.get("approximate", "none")
                flops = args[0].numel() * (18 if approximation == "tanh" else 11)
            elif name == "native_layer_norm_backward":
                # Reuse saved mean/rstd. Count only requested dx, dweight and dbias.
                x, shape, weight, mask = args[1], args[2], args[5], args[7]
                elements = x.numel()
                width = prod(shape)
                rows = elements // width
                flops = 2 * elements if mask[0] or mask[1] else 0  # xhat
                if mask[0]:
                    # g=dy*weight; two row means; dx=rstd*(g-mean(g)-xhat*mean(g*xhat)).
                    flops += (7 + int(weight is not None)) * elements
                if mask[1]:
                    flops += elements + (elements - width)
                if mask[2]:
                    flops += elements - width
            elif name in {"_log_softmax", "log_softmax"}:
                flops = 4 * args[0].numel()
            elif name == "_log_softmax_backward_data":
                grad, dim = args[0], args[2]
                flops = 4 * grad.numel() - grad.numel() // grad.shape[dim]
            elif name == "_softmax_backward_data":
                grad, dim = args[0], args[2]
                flops = 4 * grad.numel() - grad.numel() // grad.shape[dim]
            elif name in {"nll_loss_forward", "nll_loss_backward"}:
                offset = int(name == "nll_loss_backward")
                weight = args[2 + offset] if len(args) > 2 + offset else kwargs.get("weight")
                reduction = args[3 + offset] if len(args) > 3 + offset else kwargs.get("reduction", 1)
                targets = args[1 + offset].numel()
                if weight is not None:
                    self.uncovered_ops.add(str(packet) + " (weighted NLL not covered)")
                    return out
                # Current dataset has valid class IDs, no ignored targets.
                if offset:
                    flops = targets * (1 + int(reduction == 1))
                else:
                    flops = targets
                    if reduction != 0:
                        flops += max(targets - 1, 0) + int(reduction == 1)
            elif name.startswith("_foreach_"):
                op = name[len("_foreach_"):].rstrip("_")
                tensors = outputs or [x for x in tree_leaves(args[0])
                                      if isinstance(x, torch.Tensor) and x.is_floating_point()]
                elements = sum(x.numel() for x in tensors)
                if op in {"mul", "div", "sqrt", "rsqrt", "reciprocal", "neg", "pow"}:
                    flops = elements
                elif op in {"add", "sub"}:
                    flops = elements * (1 + int(kwargs.get("alpha", 1) != 1))
                elif op == "lerp":
                    flops = 3 * elements
                elif op in {"addcmul", "addcdiv"}:
                    value = args[3] if len(args) > 3 else kwargs.get("value", 1)
                    if isinstance(value, (list, tuple)):
                        flops = sum(t.numel() * (2 + int(v != 1)) for t, v in zip(tensors, value))
                    elif isinstance(value, torch.Tensor):
                        # Tensor scale values cannot be read without a device synchronization.
                        self.uncovered_ops.add(str(packet) + " (tensor scale not covered)")
                        return out
                    else:
                        flops = elements * (2 + int(value != 1))
                elif op in {"copy", "zero", "maximum", "minimum"}:
                    return out
                else:
                    self.uncovered_ops.add(str(packet))
                    return out
            elif name in {"lerp", "lerp_"}:
                flops = 3 * n
            elif name in {"addcmul", "addcmul_", "addcdiv", "addcdiv_"}:
                flops = n * (2 + int(kwargs.get("value", 1) != 1))
            elif name in {"sqrt_", "rsqrt_", "reciprocal_", "neg_", "pow_"}:
                flops = n
            else:
                # Also flag unknown mutating operators whose return value is empty.
                if not outputs and name not in self.zero_flop_ops:
                    if any(isinstance(x, torch.Tensor) and x.is_floating_point()
                           for x in tree_leaves(args)):
                        self.uncovered_ops.add(str(packet))
                    return out
                return super()._count_flops(packet, out, args, kwargs)

            self.flop_counts["Global"][packet] += int(flops)
            return out

    TrainingFlopCounter.__qualname__ = 'TrainingFlopCounter'


    def new_training_cost():
        return {}


    def add_training_cost(cost, counter, component, num_examples, input_ids=None):
        item = cost.setdefault(component, {
            "counted_flops": 0, "calls": 0, "num_examples": 0,
            "operators": defaultdict(int), "uncovered": set(),
            "sequence_lengths": set(), "batch_sizes": set(),
        })
        item["counted_flops"] += int(counter.get_total_flops())
        item["calls"] += 1
        item["num_examples"] += int(num_examples)
        item["batch_sizes"].add(int(num_examples))
        item["uncovered"].update(counter.uncovered_ops)
        for op, value in counter.get_flop_counts().get("Global", {}).items():
            item["operators"][str(op)] += int(value)
        if input_ids is not None:
            item["sequence_lengths"].add(int(input_ids.shape[1]))


    def merge_training_cost(total, addition):
        for name, item in addition.items():
            if name not in total:
                total[name] = {
                    "counted_flops": 0, "calls": 0, "num_examples": 0,
                    "operators": defaultdict(int), "uncovered": set(),
                    "sequence_lengths": set(), "batch_sizes": set(),
                }
            target = total[name]
            for key in ("counted_flops", "calls", "num_examples"):
                target[key] += item[key]
            for key in ("uncovered", "sequence_lengths", "batch_sizes"):
                target[key].update(item[key])
            for op, value in item["operators"].items():
                target["operators"][op] += value


    def finish_training_cost(cost):
        components = {}
        for name, item in cost.items():
            components[name] = {
                "total_flops": item["counted_flops"] if not item["uncovered"] else None,
                "counted_flops": item["counted_flops"],
                "calls": item["calls"],
                "num_examples": item["num_examples"],
                "counted_flops_by_operator": dict(sorted(item["operators"].items())),
                "uncovered_operators": sorted(item["uncovered"]),
                "observed_sequence_lengths": sorted(item["sequence_lengths"]),
                "observed_batch_sizes": sorted(item["batch_sizes"]),
            }
        training = [v for k, v in components.items() if "validation" not in k]
        validation = [v for k, v in components.items() if "validation" in k]

        def subtotal(items):
            if any(item["total_flops"] is None for item in items):
                return None
            return sum(item["total_flops"] for item in items)

        uncovered = sorted({op for item in components.values() for op in item["uncovered_operators"]})
        return {
            "training_flops": subtotal(training),
            "validation_flops": subtotal(validation),
            "total_flops": subtotal(list(components.values())),
            "components": components,
            "coverage": {
                "status": "incomplete" if uncovered else "complete_for_convention",
                "uncovered_operators": uncovered,
                "counted_flops": sum(item["counted_flops"] for item in components.values()),
            },
        }



    def finish_multitask_training_cost(cost_by_task):
        combined = new_training_cost()
        for cost in cost_by_task.values():
            merge_training_cost(combined, cost)
        report = finish_training_cost(combined)
        report["by_task"] = {task: finish_training_cost(cost) for task, cost in cost_by_task.items()}
        return report


    # Router-specific arithmetic. Matmuls are already counted by the base counter.
    # Fused GRU formulas follow PyTorch's gate kernels and bias reductions:
    # https://github.com/pytorch/pytorch/blob/v2.11.0/aten/src/ATen/native/cuda/RNN.cu
    class RoutedFlopCounter(TrainingFlopCounter):
        zero_flop_ops = TrainingFlopCounter.zero_flop_ops | {
            "one_hot", "scatter", "scatter_", "index_put", "index_put_",
            "_index_put_impl_", "new_empty_strided", "empty_permuted",
            "clamp", "clamp_", "clamp_min", "clamp_min_", "clamp_max", "clamp_max_",
            "exponential", "exponential_", "bernoulli", "bernoulli_",
        }

        def _count_flops(self, packet, out, args, kwargs):
            name = str(packet).split(".")[-1]
            outputs = [x for x in tree_leaves(out)
                       if isinstance(x, torch.Tensor) and x.is_floating_point()]
            if name in {"index_put", "index_put_", "_index_put_impl_"}:
                accumulate = args[3] if len(args) > 3 else kwargs.get("accumulate", False)
                if accumulate:
                    self.flop_counts["Global"][packet] += args[2].numel()
                return out
            if name in {"sigmoid", "sigmoid_"}:
                flops = 4 * args[0].numel()  # negation, exp, addition, reciprocal
            elif name == "sigmoid_backward":
                flops = 3 * args[0].numel()
            elif name == "_thnn_fused_gru_cell":
                # input/hidden gate matmuls are separate; the fused pointwise kernel
                # includes both biases, two sigmoids, tanh and the saved hidden gate.
                if len(args) < 5 or args[3] is None or args[4] is None:
                    self.uncovered_ops.add(str(packet) + " (bias-free GRU not covered)")
                    return out
                flops = 23 * args[2].numel()
            elif name == "_thnn_fused_gru_cell_backward":
                grad = args[0]
                has_bias = args[2] if len(args) > 2 else kwargs.get("has_bias", False)
                flops = 16 * grad.numel()
                if has_bias:
                    flops += 6 * grad.shape[-1] * max(grad.shape[0] - 1, 0)
            elif name in {"std", "var", "std_mean", "var_mean"}:
                # Mean + centered squares + reduction/division (+ sqrt for std).
                n = args[0].numel()
                rows = outputs[0].numel()
                flops = 4 * n + (rows if name.startswith("std") else 0)
            elif name == "linalg_vector_norm":
                order = args[1] if len(args) > 1 else kwargs.get("ord", 2)
                if order != 2:
                    self.uncovered_ops.add(str(packet) + " (non-L2 norm not covered)")
                    return out
                flops = 2 * args[0].numel()  # square + sum + sqrt
            elif name == "_foreach_norm":
                order = args[1] if len(args) > 1 else kwargs.get("ord", 2)
                if order != 2:
                    self.uncovered_ops.add(str(packet) + " (non-L2 norm not covered)")
                    return out
                flops = 2 * sum(t.numel() for t in args[0])
            elif name in {"log_", "exp_"}:
                flops = args[0].numel()
            elif name in {
                "one_hot", "scatter", "scatter_", "clamp", "clamp_", "clamp_min", "clamp_min_",
                "clamp_max", "clamp_max_", "exponential", "exponential_",
            }:
                return out
            else:
                return super()._count_flops(packet, out, args, kwargs)
            self.flop_counts["Global"][packet] += int(flops)
            return out

    RoutedFlopCounter.__qualname__ = 'RoutedFlopCounter'


    def finish_routed_training_cost(cost_by_task):
        result = finish_multitask_training_cost(cost_by_task)
        result["method"] = "pytorch_executed_training_flops_v1"
        result["scope"] = (
            "all completed epochs: actual forward/loss, backward through frozen layers "
            "where required, gradient clipping and optimizer tensor arithmetic; "
            "validation reported separately; prior expert training excluded"
        )
        result["convention"] = {
            "multiply_accumulate_flops": 2,
            "elementwise_arithmetic_or_special_function_flops": 1,
            "sigmoid_flops_per_element": 4,
            "router": "shared MLP executed matmuls, biases, GELU forward/backward and depth embedding gradients",
            "attention": "same dense attention convention as the other evaluations; fused backward includes recomputation",
            "excluded": "RNG generation, Python scalar bookkeeping, metrics, diagnostics, preprocessing, I/O, copies, indexing, comparisons",
            "interpretation": "formula-based FLOPs of executed operators, not exact GPU kernel instructions",
        }
        result["torch_version"] = str(torch.__version__)
        result["transformers_version"] = transformers_version
        return result


    def json_config_value(value):
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, torch.device):
            return str(value)
        if isinstance(value, dict):
            return {str(k): json_config_value(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [json_config_value(v) for v in value]
        if isinstance(value, float) and not isfinite(value):
            return str(value)
        if isinstance(value, (str, int, float, bool)) or value is None:
            return value
        return str(value)


In [ ]:
def prepare_dataloaders():
    global BATCH_SIZE, DataLoader, MAX_LEN, NUM_WORKERS, PIN_MEMORY, TensorDataset, mixed_batch_iterator, mnli_test_dataset
    global mnli_test_loader, mnli_train_dataset, mnli_train_loader, mnli_val_dataset, mnli_val_loader, random, sample_batch, sample_task
    global sst_test_dataset, sst_test_loader, sst_train_dataset, sst_train_loader, sst_val_dataset, sst_val_loader, tokenize_texts, torch
    # ============================================================
    # TOKENIZATION, DATALOADERS, AND RANDOM MIXED BATCHING
    # ============================================================

    import torch
    import random
    from torch.utils.data import TensorDataset, DataLoader

    # ============================================================
    # TOKENIZATION PARAMETERS
    # ============================================================

    MAX_LEN = 256
    BATCH_SIZE = 32
    NUM_WORKERS = 0
    PIN_MEMORY = torch.cuda.is_available()

    # ============================================================
    # TOKENIZATION FUNCTION
    # ============================================================

    def tokenize_texts(texts, labels, tokenizer, max_len, task):
        # Prepare MNLI as a sentence pair, consistently with the MNLI expert.
        text_pairs = None
        if task == "mnli":
            premises = []
            hypotheses = []
            for text in texts:
                parts = str(text).split("[SEP]", 1)
                premises.append(parts[0].strip())
                hypotheses.append(parts[1].strip() if len(parts) > 1 else "")
            texts = premises
            text_pairs = hypotheses

        encodings = tokenizer(
            texts,
            text_pair=text_pairs,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
            return_token_type_ids=True
        )

        input_ids = encodings["input_ids"]
        attention_mask = encodings["attention_mask"]
        token_type_ids = encodings.get("token_type_ids", torch.zeros_like(input_ids))
        labels = torch.tensor(labels, dtype=torch.long)

        return TensorDataset(
            input_ids,
            attention_mask,
            token_type_ids,
            labels
        )

    # ============================================================
    # CREATE TENSOR DATASETS
    # ============================================================

    print("Tokenizing SST...")

    sst_train_dataset = tokenize_texts(x_sst_train_run, y_sst_train_run, tokenizer, MAX_LEN, task="sst")
    sst_val_dataset   = tokenize_texts(x_sst_val_run,   y_sst_val_run,   tokenizer, MAX_LEN, task="sst")
    sst_test_dataset  = tokenize_texts(x_sst_test_run,  y_sst_test_run,  tokenizer, MAX_LEN, task="sst")

    print("Tokenizing MNLI...")

    mnli_train_dataset = tokenize_texts(x_mnli_train_run, y_mnli_train_run, tokenizer, MAX_LEN, task="mnli")
    mnli_val_dataset   = tokenize_texts(x_mnli_val_run,   y_mnli_val_run,   tokenizer, MAX_LEN, task="mnli")
    mnli_test_dataset  = tokenize_texts(x_mnli_test_run,  y_mnli_test_run,  tokenizer, MAX_LEN, task="mnli")

    # ============================================================
    # CREATE DATALOADERS
    # ============================================================

    sst_train_loader = DataLoader(
        sst_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    mnli_train_loader = DataLoader(
        mnli_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    sst_val_loader = DataLoader(
        sst_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    mnli_val_loader = DataLoader(
        mnli_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    sst_test_loader = DataLoader(
        sst_test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    mnli_test_loader = DataLoader(
        mnli_test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    # ============================================================
    # RANDOM MIXED BATCH ITERATOR
    # ============================================================

    def mixed_batch_iterator(sst_loader, mnli_loader, seed=None):
        rng = random.Random(seed)

        sst_iter = iter(sst_loader)
        mnli_iter = iter(mnli_loader)

        remaining = {
            "sst": len(sst_loader),
            "mnli": len(mnli_loader)
        }

        while remaining["sst"] > 0 or remaining["mnli"] > 0:

            available_tasks = [
                task for task, count in remaining.items()
                if count > 0
            ]

            task = rng.choice(available_tasks)

            if task == "sst":
                batch = next(sst_iter)
                remaining["sst"] -= 1

            else:
                batch = next(mnli_iter)
                remaining["mnli"] -= 1

            yield task, batch

    # ============================================================
    # PRINT CHECK
    # ============================================================

    print("=" * 70)
    print("DATALOADERS CREATED")
    print("=" * 70)

    print("MAX_LEN:", MAX_LEN)
    print("BATCH_SIZE:", BATCH_SIZE)

    print("\nSST")
    print("Train batches:", len(sst_train_loader))
    print("Val batches:  ", len(sst_val_loader))
    print("Test batches: ", len(sst_test_loader))

    print("\nMNLI")
    print("Train batches:", len(mnli_train_loader))
    print("Val batches:  ", len(mnli_val_loader))
    print("Test batches: ", len(mnli_test_loader))

    print("\nMixed train batches per epoch:", len(sst_train_loader) + len(mnli_train_loader))

    sample_task, sample_batch = next(
        mixed_batch_iterator(
            sst_train_loader,
            mnli_train_loader,
            seed=SEED
        )
    )

    print("\nSample mixed batch task:", sample_task)
    print("input_ids:     ", sample_batch[0].shape)
    print("attention_mask:", sample_batch[1].shape)
    print("token_type_ids:", sample_batch[2].shape)
    print("labels:        ", sample_batch[3].shape)

In [ ]:
def configure_loss():
    global CLASSIFICATION_LOSS_WEIGHT, ENTROPY_BONUS_WEIGHT_END, ENTROPY_BONUS_WEIGHT_START, ENTROPY_RAMP_END_FRACTION, MNLI_COMPUTE_COST_WEIGHT_END, MNLI_COMPUTE_COST_WEIGHT_START, MNLI_COST_RAMP_END_FRACTION, ROUTER_LOSS_EPS
    global SST_COMPUTE_COST_WEIGHT, classification_criterion, compute_total_loss, loss_weights, math, nn, schedule_endpoint, scheduled_weight
    global torch
    # ============================================================
    # LOSS: classification + task-specific cost - temporary entropy
    # ============================================================

    import math
    import torch
    import torch.nn as nn

    CLASSIFICATION_LOSS_WEIGHT = 1.15
    SST_COMPUTE_COST_WEIGHT = 0.035
    MNLI_COMPUTE_COST_WEIGHT_START = 0.020
    MNLI_COMPUTE_COST_WEIGHT_END = 0.030
    MNLI_COST_RAMP_END_FRACTION = 5.0 / 8.0
    ENTROPY_BONUS_WEIGHT_START = 0.005
    ENTROPY_BONUS_WEIGHT_END = 0.0
    ENTROPY_RAMP_END_FRACTION = 1.0
    ROUTER_LOSS_EPS = 1e-8

    classification_criterion = nn.CrossEntropyLoss()


    def schedule_endpoint(total_epochs, fraction):
        """One-based endpoint: e.g. 8 epochs -> 8 (entropy), 5 (MNLI cost)."""
        if type(total_epochs) is not int or total_epochs < 1:
            raise ValueError("EPOCHS must be an integer >= 1")
        if not 0 < fraction <= 1:
            raise ValueError("Schedule fractions must be in (0, 1]")
        return min(total_epochs, max(2, math.ceil(total_epochs * fraction)))


    def scheduled_weight(epoch, total_epochs, start, end, fraction):
        if not 0 <= epoch < total_epochs:
            raise ValueError("Epoch index outside configured training")
        endpoint = schedule_endpoint(total_epochs, fraction)
        if endpoint == 1:
            return float(end)  # One epoch: use final weights and final temperature.
        progress = min(epoch / (endpoint - 1), 1.0)
        return float(start + progress * (end - start))


    def loss_weights(task, epoch, total_epochs):
        if task not in ("sst", "mnli"):
            raise ValueError("Unknown task")
        cost_weight = (
            SST_COMPUTE_COST_WEIGHT if task == "sst" else
            scheduled_weight(epoch, total_epochs, MNLI_COMPUTE_COST_WEIGHT_START,
                             MNLI_COMPUTE_COST_WEIGHT_END, MNLI_COST_RAMP_END_FRACTION)
        )
        entropy_weight = scheduled_weight(
            epoch, total_epochs, ENTROPY_BONUS_WEIGHT_START,
            ENTROPY_BONUS_WEIGHT_END, ENTROPY_RAMP_END_FRACTION)
        return {"classification_loss_weight": CLASSIFICATION_LOSS_WEIGHT,
                "compute_cost_weight": float(cost_weight),
                "entropy_bonus_weight": entropy_weight}


    def compute_total_loss(outputs, labels, task, epoch, total_epochs):
        weights = loss_weights(task, epoch, total_epochs)
        classification_loss = classification_criterion(outputs["logits"], labels)
        probabilities = outputs["routing_probs"]
        # Sum across layers, mean across examples: scale remains 0..12, as before.
        expected_active_layers = (1.0 - probabilities[:, :, CHOICE_IDENTITY]).sum(dim=1).mean()
        # Unavailable MERGED has probability exactly zero after the router mask.
        # log(p + eps) keeps that term finite: 0 * log(eps) contributes zero.
        # No division by log(number of choices): keep the first trial's loss scale.
        entropy_bonus = -(
            probabilities * torch.log((probabilities + ROUTER_LOSS_EPS))
        ).sum(dim=-1).mean()
        compute_cost_loss = weights["compute_cost_weight"] * expected_active_layers
        entropy_regularization = -weights["entropy_bonus_weight"] * entropy_bonus
        total_loss = (
            weights["classification_loss_weight"] * classification_loss
            + compute_cost_loss + entropy_regularization
        )
        return {
            "total_loss": total_loss, "classification_loss": classification_loss,
            "expected_active_layers": expected_active_layers,
            "compute_cost_loss": compute_cost_loss,
            "entropy_bonus": entropy_bonus,
            "entropy_regularization": entropy_regularization,
            **weights,
        }


    print("Loss: cross-entropy + task-specific expected active layers - temporary entropy")
    print("SST cost weight:", SST_COMPUTE_COST_WEIGHT)
    print("MNLI cost weight:", MNLI_COMPUTE_COST_WEIGHT_START, "->", MNLI_COMPUTE_COST_WEIGHT_END)
    print("Entropy weight:", ENTROPY_BONUS_WEIGHT_START, "->", ENTROPY_BONUS_WEIGHT_END)


In [ ]:
def train_routed():
    global BEST_EFFICIENCY_MODEL_PATH, CHOICE_NAMES, EARLY_STOPPING_ENABLED, EFFICIENT_EVAL, EPOCHS, EVALUATION_CHECKPOINT, ROUTED_MODEL_SAVE_PATH, SHOW_BATCH_PROGRESS
    global TAU_ANNEALING_EPOCHS, TAU_END, TAU_START, accuracy_score, best_efficiency_checkpoint, best_epoch, best_quality_checkpoint, best_routing_stats
    global best_val_f1_avg, best_validation_metrics, compute_metrics, criterion, current_lrs, current_training_cost, efficiency, efficiency_candidate
    global epoch, epoch_cost_by_task, epoch_record, evaluate_original_expert_reference, evaluate_task, experiment_config, f, finalize_routing_counter
    global get_optimizer_lrs, get_tau, global_decision_counts, global_example_counts, group, init_routing_counter, is_best, is_best_efficiency
    global json, key, loader, loaders, merge_routing_counts, metrics_by_task, mnli_routing_stats, mnli_val_metrics
    global move_batch_to_device, nn, np, old_efficiency, precision_recall_fscore_support, print_epoch_compact, reference, routing_epoch_record
    global routing_history, save_best_model, save_training_summary, selection_cost, split, sst_routing_stats, sst_val_metrics, start_time
    global t, task, tau, time, torch, tqdm, train_metrics, train_one_epoch
    global training_cost_by_task, training_history, training_summary, update_routing_counter, val_acc_avg, val_f1_avg, val_loss_avg, validation_references
    global value
    # ============================================================
    # TRAINING LOOP - ROUTED MODEL
    # Full epoch budget; save best quality and best admissible efficiency
    # ============================================================

    import json
    import time
    import torch
    import torch.nn as nn
    import numpy as np

    from tqdm.auto import tqdm
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support

    # ============================================================
    # TRAINING PARAMETERS
    # ============================================================

    EPOCHS = 8
    if type(EPOCHS) is not int or EPOCHS < 1:
        raise ValueError("EPOCHS must be an integer >= 1")

    # One model is exported/evaluated; both checkpoint files remain available.
    EVALUATION_CHECKPOINT = "efficiency_if_available"  # or "quality"
    EARLY_STOPPING_ENABLED = False  # Complete the configured budget and schedules.

    TAU_START = 1.0
    TAU_END = 0.1
    TAU_ANNEALING_EPOCHS = EPOCHS

    EFFICIENT_EVAL = True
    SHOW_BATCH_PROGRESS = True

    criterion = classification_criterion

    CHOICE_NAMES = ["SST", "MNLI", "MERGED", "IDENTITY"]

    MODELS_PATH.mkdir(parents=True, exist_ok=True)
    ROUTED_MODEL_SAVE_PATH = MODELS_PATH / "routed_model.pt"  # best mean validation F1
    BEST_EFFICIENCY_MODEL_PATH = MODELS_PATH / "routed_model_best_efficiency.pt"


    # ============================================================
    # HELPER FUNCTIONS
    # ============================================================

    def get_tau(epoch, tau_start=1.0, tau_end=0.1, annealing_epochs=None):
        if annealing_epochs is None:
            annealing_epochs = EPOCHS
        if type(annealing_epochs) is not int or annealing_epochs < 1:
            raise ValueError("Annealing length must be an integer >= 1")
        if not 0 <= epoch < annealing_epochs:
            raise ValueError("Epoch outside temperature schedule")
        if annealing_epochs <= 1:
            return tau_end

        progress = epoch / (annealing_epochs - 1)
        progress = min(progress, 1.0)

        tau = tau_start + progress * (tau_end - tau_start)
        return float(tau)


    def move_batch_to_device(batch, device):
        input_ids, attention_mask, token_type_ids, labels = batch

        return (
            input_ids.to(device),
            attention_mask.to(device),
            token_type_ids.to(device),
            labels.to(device)
        )


    def get_optimizer_lrs(optimizer):
        lr_dict = {}

        for group in optimizer.param_groups:
            group_name = group.get("name", "unnamed")
            lr_dict[group_name] = float(group["lr"])

        return lr_dict


    def compute_metrics(y_true, y_pred, task):
        average = "binary" if task == "sst" else "macro"

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average=average,
            zero_division=0
        )

        accuracy = accuracy_score(y_true, y_pred)

        return {
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1)
        }


    # ============================================================
    # ROUTING COUNTERS
    # ============================================================

    def init_routing_counter(num_layers=12, num_choices=4):
        return {
            "choice_counts": np.zeros((num_layers, num_choices), dtype=np.float64),
            "soft_prob_sums": np.zeros((num_layers, num_choices), dtype=np.float64),
            "examples_with_choice": np.zeros(num_choices, dtype=np.float64),
            "total_examples": 0,
            "active_layers_sum": 0.0
        }


    def update_routing_counter(counter, routing_choices, routing_probs):
        choices_np = routing_choices.detach().cpu().numpy()
        probs_np = routing_probs.detach().cpu().numpy()

        batch_size = choices_np.shape[0]

        counter["choice_counts"] += choices_np.sum(axis=0)
        counter["soft_prob_sums"] += probs_np.sum(axis=0)
        counter["total_examples"] += batch_size

        choice_idx = choices_np.argmax(axis=2)
        active_layers = (choice_idx != CHOICE_IDENTITY).sum(axis=1)

        counter["active_layers_sum"] += active_layers.sum()

        used_choice_at_least_once = choices_np.sum(axis=1) > 0
        counter["examples_with_choice"] += used_choice_at_least_once.sum(axis=0)


    def finalize_routing_counter(counter):
        total_examples = counter["total_examples"]

        if total_examples == 0:
            raise ValueError("No examples found for routing analysis.")

        hard_choice_fractions = counter["choice_counts"] / total_examples
        soft_prob_fractions = counter["soft_prob_sums"] / total_examples
        avg_active_layers = counter["active_layers_sum"] / total_examples

        total_choice_counts = counter["choice_counts"].sum(axis=0)

        return {
            "hard_choice_fractions": hard_choice_fractions.tolist(),
            "soft_prob_fractions": soft_prob_fractions.tolist(),

            "total_choice_counts": {
                "SST": int(total_choice_counts[CHOICE_SST]),
                "MNLI": int(total_choice_counts[CHOICE_MNLI]),
                "MERGED": int(total_choice_counts[CHOICE_MERGED]),
                "IDENTITY": int(total_choice_counts[CHOICE_IDENTITY])
            },

            "examples_with_choice": {
                "SST": int(counter["examples_with_choice"][CHOICE_SST]),
                "MNLI": int(counter["examples_with_choice"][CHOICE_MNLI]),
                "MERGED": int(counter["examples_with_choice"][CHOICE_MERGED]),
                "IDENTITY": int(counter["examples_with_choice"][CHOICE_IDENTITY])
            },

            "routing_availability": ROUTING_AVAILABILITY,
            "avg_active_layers": float(avg_active_layers),
            "num_examples": int(total_examples)
        }


    # ============================================================
    # EVALUATION DURING TRAINING
    # ============================================================

    def evaluate_task(model, dataloader, task, criterion, device, return_routing_stats=True, cost=None):
        set_routed_model_eval_mode(model)

        total_loss = 0.0
        total_samples = 0

        all_preds = []
        all_labels = []
        inference_cost = new_inference_cost()

        routing_counter = init_routing_counter(
            num_layers=NUM_LAYERS,
            num_choices=NUM_CHOICES
        )

        with torch.no_grad():
            for batch in dataloader:
                input_ids, attention_mask, token_type_ids, labels = move_batch_to_device(batch, device)

                with RoutedFlopCounter() as validation_counter:
                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids,
                        task=task,
                        tau=TAU_END,
                        efficient_inference=EFFICIENT_EVAL,
                        return_routing=return_routing_stats
                    )

                    logits = outputs["logits"] if return_routing_stats else outputs

                add_inference_cost(inference_cost, validation_counter, "routed_model", input_ids)
                with RoutedFlopCounter() as validation_loss_counter:
                    loss = criterion(logits, labels)
                if cost is not None:
                    add_training_cost(cost, validation_counter, "validation_forward", labels.size(0), input_ids)
                    add_training_cost(cost, validation_loss_counter, "validation_loss", labels.size(0))

                if return_routing_stats:
                    logits = outputs["logits"]
                    update_routing_counter(
                        routing_counter,
                        outputs["routing_choices"],
                        outputs["routing_probs"]
                    )
                else:
                    logits = outputs

                preds = torch.argmax(logits, dim=1)

                batch_size = labels.size(0)

                total_loss += loss.item() * batch_size
                total_samples += batch_size

                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())

        metrics = compute_metrics(all_labels, all_preds, task)
        metrics["loss"] = float(total_loss / total_samples)
        metrics["classification_loss"] = float(total_loss / total_samples)
        metrics["num_samples"] = int(total_samples)
        metrics["inference_cost"] = finish_inference_cost(inference_cost, total_samples)
        metrics["inference_cost"]["scope"] = "validation forward only: inputs to logits; excludes loss/metrics"

        if return_routing_stats:
            routing_stats = finalize_routing_counter(routing_counter)
            return metrics, routing_stats

        return metrics, None


    # ============================================================
    # TRAIN ONE EPOCH
    # ============================================================

    def train_one_epoch(model, sst_loader, mnli_loader, optimizer, device, epoch, tau, cost_by_task):
        set_routed_model_train_mode(model)

        task_storage = {
            "sst": {
                "total_loss": 0.0,
                "classification_loss": 0.0,
                "expected_active_layers": 0.0,
                "compute_cost_loss": 0.0,
                "entropy_bonus": 0.0,
                "entropy_regularization": 0.0,
                "samples": 0,
                "preds": [],
                "labels": []
            },
            "mnli": {
                "total_loss": 0.0,
                "classification_loss": 0.0,
                "expected_active_layers": 0.0,
                "compute_cost_loss": 0.0,
                "entropy_bonus": 0.0,
                "entropy_regularization": 0.0,
                "samples": 0,
                "preds": [],
                "labels": []
            }
        }

        total_batches = len(sst_loader) + len(mnli_loader)

        progress_bar = tqdm(
            mixed_batch_iterator(
                sst_loader,
                mnli_loader,
                seed=SEED + epoch
            ),
            total=total_batches,
            desc=f"Epoch {epoch + 1}/{EPOCHS}",
            disable=not SHOW_BATCH_PROGRESS,
            leave=True
        )

        for task, batch in progress_bar:
            input_ids, attention_mask, token_type_ids, labels = move_batch_to_device(batch, device)

            optimizer.zero_grad(set_to_none=True)

            with RoutedFlopCounter(count_attention_saved_stats=True) as forward_counter:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                    task=task,
                    tau=tau,
                    efficient_inference=False,
                    return_routing=True
                )

                loss_dict = compute_total_loss(
                    outputs=outputs, labels=labels,
                    task=task, epoch=epoch, total_epochs=EPOCHS
                )

                loss = loss_dict["total_loss"]
                logits = outputs["logits"]

            add_training_cost(cost_by_task[task], forward_counter, "model_forward_and_loss", labels.size(0), input_ids)

            with RoutedFlopCounter() as backward_counter:
                loss.backward()
            add_training_cost(cost_by_task[task], backward_counter, "backward", labels.size(0), input_ids)

            if GRADIENT_CLIPPING:
                with RoutedFlopCounter() as clipping_counter:
                    torch.nn.utils.clip_grad_norm_(
                        [p for p in model.parameters() if p.requires_grad],
                        MAX_GRAD_NORM
                    )
                add_training_cost(cost_by_task[task], clipping_counter, "gradient_clipping", labels.size(0))

            with RoutedFlopCounter() as optimizer_counter:
                optimizer.step()
            add_training_cost(cost_by_task[task], optimizer_counter, "optimizer_step", labels.size(0))

            preds = torch.argmax(logits, dim=1)
            batch_size = labels.size(0)

            task_storage[task]["total_loss"] += loss_dict["total_loss"].item() * batch_size
            task_storage[task]["classification_loss"] += loss_dict["classification_loss"].item() * batch_size
            task_storage[task]["expected_active_layers"] += loss_dict["expected_active_layers"].item() * batch_size
            task_storage[task]["compute_cost_loss"] += loss_dict["compute_cost_loss"].item() * batch_size
            task_storage[task]["entropy_bonus"] += loss_dict["entropy_bonus"].item() * batch_size
            task_storage[task]["entropy_regularization"] += loss_dict["entropy_regularization"].item() * batch_size

            task_storage[task]["samples"] += batch_size
            task_storage[task]["preds"].extend(preds.detach().cpu().tolist())
            task_storage[task]["labels"].extend(labels.detach().cpu().tolist())

            if SHOW_BATCH_PROGRESS:
                progress_bar.set_postfix({
                    "task": task,
                    "loss": f"{loss.item():.4f}",
                    "tau": f"{tau:.2f}"
                })

        epoch_metrics = {}

        for task in ["sst", "mnli"]:
            samples = task_storage[task]["samples"]

            task_metrics = compute_metrics(
                task_storage[task]["labels"],
                task_storage[task]["preds"],
                task
            )

            task_metrics["loss"] = float(task_storage[task]["classification_loss"] / samples)
            task_metrics["total_loss"] = float(task_storage[task]["total_loss"] / samples)
            task_metrics["classification_loss"] = float(task_storage[task]["classification_loss"] / samples)
            task_metrics["expected_active_layers"] = float(task_storage[task]["expected_active_layers"] / samples)
            task_metrics["compute_cost_loss"] = float(task_storage[task]["compute_cost_loss"] / samples)
            task_metrics["entropy_bonus"] = float(task_storage[task]["entropy_bonus"] / samples)
            task_metrics["entropy_regularization"] = float(task_storage[task]["entropy_regularization"] / samples)
            task_metrics["loss_weights"] = loss_weights(task, epoch, EPOCHS)
            task_metrics["num_samples"] = int(samples)

            epoch_metrics[task] = task_metrics

        return epoch_metrics


    # ============================================================
    # PRINT AND SAVE HELPERS
    # ============================================================

    def merge_routing_counts(sst_routing_stats, mnli_routing_stats, key_name):
        sst_counts = sst_routing_stats[key_name]
        mnli_counts = mnli_routing_stats[key_name]

        return {
            "SST": sst_counts["SST"] + mnli_counts["SST"],
            "MNLI": sst_counts["MNLI"] + mnli_counts["MNLI"],
            "MERGED": sst_counts["MERGED"] + mnli_counts["MERGED"],
            "IDENTITY": sst_counts["IDENTITY"] + mnli_counts["IDENTITY"]
        }


    def print_epoch_compact(
        epoch,
        epochs,
        tau,
        train_metrics,
        sst_val_metrics,
        mnli_val_metrics,
        val_f1_avg,
        best
    ):
        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"tau={tau:.3f} | "
            f"val_f1_avg={val_f1_avg:.4f} | "
            f"best={best}"
        )

        print(
            f"  SST  train_f1={train_metrics['sst']['f1']:.4f} "
            f"val_f1={sst_val_metrics['f1']:.4f} "
            f"val_acc={sst_val_metrics['accuracy']:.4f}"
        )

        print(
            f"  MNLI train_f1={train_metrics['mnli']['f1']:.4f} "
            f"val_f1={mnli_val_metrics['f1']:.4f} "
            f"val_acc={mnli_val_metrics['accuracy']:.4f}"
        )


    def save_best_model(model, save_path):
        validate_merged_layout(model)
        atomic_save_file(save_path, lambda handle: torch.save(model, handle))


    # ============================================================
    # TRAINING
    # ============================================================


    # ============================================================
    # FIXED VALIDATION REFERENCES AND CHECKPOINT SELECTION
    # ============================================================

    def selection_cost(cost):
        """Comparable forward-only FLOPs; never include classification loss."""
        coverage = cost["coverage"]
        missing = set(coverage.get("uncovered_operators", []))
        n = cost["num_examples"]
        if coverage["status"] == "complete_for_convention":
            value = cost["flops_per_example"]
        elif missing == {"aten.constant_pad_nd"}:
            value = coverage["counted_flops"] / n
        else:
            return None
        return float(value) if value is not None and math.isfinite(value) and value > 0 else None


    @torch.no_grad()
    def evaluate_original_expert_reference(backbone, classifier_path, dataloader, task):
        # ClassifierHead matches Baselines/classifier.py exactly, including state keys.
        original_head = ClassifierHead(
            input_dim=HIDDEN_SIZE, hidden_dim=HEAD_HIDDEN_SIZE,
            num_classes=2 if task == "sst" else 3,
            dropout_prob=DROPOUT_PROBABILITY,
        ).to(DEVICE)
        original_head.load_state_dict(
            torch.load(classifier_path, map_location=DEVICE, weights_only=True), strict=True)
        original_head.eval()
        backbone.eval()
        cost = new_inference_cost()
        true, predicted = [], []
        for batch in tqdm(dataloader, desc=f"Fixed {task.upper()} expert validation", leave=False):
            input_ids, attention_mask, token_type_ids, labels = move_batch_to_device(batch, DEVICE)
            with RoutedFlopCounter() as counter:
                output = backbone(input_ids=input_ids, attention_mask=attention_mask,
                                  token_type_ids=token_type_ids)
                logits = original_head(output.pooler_output)
            add_inference_cost(cost, counter, "original_expert", input_ids)
            true.extend(labels.cpu().tolist())
            predicted.extend(logits.argmax(dim=-1).cpu().tolist())
        metrics = compute_metrics(true, predicted, task)
        metrics["num_samples"] = len(true)
        finished = finish_inference_cost(cost, len(true))
        finished["scope"] = "Original expert validation: backbone and original head; no loss/metrics"
        finished["convention"] = dict(finished["convention"])
        finished["convention"].pop("router", None)
        value = selection_cost(finished)
        if value is None:
            raise RuntimeError(f"{task}: incomplete expert FLOPs; cannot set efficiency reference: "
                               f"{finished['coverage']['uncovered_operators']}")
        return {"metrics": metrics, "flops_per_example": value,
                "inference_cost": finished, "classifier_path": str(classifier_path),
                "backbone_path": str(SST_BACKBONE_PATH if task == "sst" else MNLI_BACKBONE_PATH),
                "split": "validation"}


    # Fix references before any optimizer step, using the exact validation loaders.
    # Preserve RNG state: creating reference heads/iterating loaders must not change training RNG.
    with torch.random.fork_rng(devices=[]):
        validation_references = {
            "sst": evaluate_original_expert_reference(
                sst_backbone, SST_EXPERT_PATH / "classifier.pt", sst_val_loader, "sst"),
            "mnli": evaluate_original_expert_reference(
                mnli_backbone, MNLI_EXPERT_PATH / "classifier.pt", mnli_val_loader, "mnli"),
        }
    print("Fixed validation references:",
          {task: {"f1": reference["metrics"]["f1"],
                  "GFLOPs": reference["flops_per_example"] / 1e9}
           for task, reference in validation_references.items()})


    def efficiency_candidate(epoch, metrics_by_task):
        f1_ok = {
            task: (
                metrics_by_task[task]["num_samples"] == validation_references[task]["metrics"]["num_samples"]
                and metrics_by_task[task]["f1"] >=
                validation_references[task]["metrics"]["f1"] - QUALITY_TOLERANCE_ABSOLUTE - 1e-12
            ) for task in ("sst", "mnli")
        }
        ratios = {}
        unavailable = []
        for task in ("sst", "mnli"):
            value = selection_cost(metrics_by_task[task]["inference_cost"])
            if value is None:
                unavailable.append(task)
            else:
                ratios[task] = value / validation_references[task]["flops_per_example"]
        mean_ratio = sum(ratios.values()) / 2 if len(ratios) == 2 else None
        return {
            "epoch": epoch + 1, "quality_within_tolerance": f1_ok,
            "eligible": all(f1_ok.values()) and mean_ratio is not None,
            "normalized_cost_by_task": ratios, "mean_normalized_cost": mean_ratio,
            "cost_unavailable_tasks": unavailable,
            "validation_metrics": metrics_by_task,
            "mean_f1": sum(metrics_by_task[t]["f1"] for t in ("sst", "mnli")) / 2,
        }


    experiment_config = {
        "method": "RoutedModel_MLP_task_cost_ramp_entropy_decay_merged_first_layer",
        "variant": MODEL_VARIANT,
        "experiment_group": experiment_group_metadata(),
        "routing_availability": ROUTING_AVAILABILITY,
        "run_started_at": RUN_STARTED_AT,
        "parameters": {
            key: json_config_value(value) for key, value in list(globals().items())
            if key.isupper() and isinstance(value, (str, int, float, bool, Path, torch.device, list, tuple))
        },
        "backbone_configs": {
            "sst": sst_backbone.config.to_dict(),
            "mnli": mnli_backbone.config.to_dict(),
            "merged": merged_backbone.config.to_dict(),
        },
        "architecture": str(routed_model),
        "optimizer": {
            "class": type(optimizer).__name__,
            "parameter_groups": [
                {key: json_config_value(value) for key, value in group.items() if key != "params"}
                for group in optimizer.param_groups
            ],
        },
        "scheduler": {"class": type(scheduler).__name__, "initial_state": json_config_value(scheduler.state_dict())},
        "input_preparation": {
            "sst": "single text", "mnli": "split first [SEP], strip both parts, tokenizer sentence pair",
            "max_len": MAX_LEN, "padding": "longest_in_split", "truncation": True,
            "split_sizes": {
                task: {split: len(loader.dataset) for split, loader in loaders.items()}
                for task, loaders in {
                    "sst": {"train": sst_train_loader, "validation": sst_val_loader, "test": sst_test_loader},
                    "mnli": {"train": mnli_train_loader, "validation": mnli_val_loader, "test": mnli_test_loader},
                }.items()
            },
        },
        "trainable_params": int(count_trainable_params(routed_model)),
        "total_params": int(count_total_params(routed_model)),
        "torch_version": str(torch.__version__), "transformers_version": transformers_version,
        "device": str(DEVICE), "dtype": str(next(routed_model.parameters()).dtype),
    }
    training_cost_by_task = {task: new_training_cost() for task in ("sst", "mnli")}
    best_epoch = None
    best_routing_stats = None
    best_validation_metrics = None
    best_quality_checkpoint = None
    best_efficiency_checkpoint = None


    def current_training_cost():
        result = finish_routed_training_cost(training_cost_by_task)
        result["preparation_cost"] = merge_cost
        result["total_with_preparation_flops"] = (
            result["total_flops"] + merge_cost["total_flops"]
            if result["total_flops"] is not None else None
        )
        result["per_epoch"] = [{"epoch": row["epoch"], **row["training_cost"]} for row in training_history]
        return result


    def save_training_summary(status):
        summary = {
            "method": "RoutedModel_MLP_task_cost_ramp_entropy_decay_merged_first_layer",
            "routing_availability": ROUTING_AVAILABILITY,
            "run_started_at": RUN_STARTED_AT,
            "updated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
            "status": status,
            "experiment": experiment_config,
            "epochs_completed": len(training_history),
            "best_epoch": best_epoch,
            "best_val_f1_avg": float(best_val_f1_avg),
            "best_validation_metrics": best_validation_metrics,
            "best_model_path": str(ROUTED_MODEL_SAVE_PATH),
            "routing_analysis_path": str(ROUTING_ANALYSIS_PATH),
            "final_routing_stats": best_routing_stats,
            "routing_stats_reference": "validation at the saved best epoch",
            "training_cost": current_training_cost(),
            "trainable_params": int(count_trainable_params(routed_model)),
            "total_params": int(count_total_params(routed_model)),

            "merge_alpha_sst": float(MERGE_ALPHA_SST),
            "merge_beta_mnli": float(MERGE_BETA_MNLI),

            "router_learning_rate": float(ROUTER_LEARNING_RATE),
            "head_learning_rate": float(HEAD_LEARNING_RATE),
            "router_weight_decay": float(ROUTER_WEIGHT_DECAY),
            "head_weight_decay": float(HEAD_WEIGHT_DECAY),

            "loss_schedule": {
                "formula": "classification + task_cost_weight * expected_active_layers - entropy_weight * entropy",
                "sst_compute_cost_weight": SST_COMPUTE_COST_WEIGHT,
                "mnli_compute_cost_weight_start": MNLI_COMPUTE_COST_WEIGHT_START,
                "mnli_compute_cost_weight_end": MNLI_COMPUTE_COST_WEIGHT_END,
                "mnli_cost_ramp_endpoint_epoch": schedule_endpoint(EPOCHS, MNLI_COST_RAMP_END_FRACTION),
                "entropy_start": ENTROPY_BONUS_WEIGHT_START,
                "entropy_end": ENTROPY_BONUS_WEIGHT_END,
                "entropy_endpoint_epoch": schedule_endpoint(EPOCHS, ENTROPY_RAMP_END_FRACTION),
                "per_epoch": [
                    {"epoch": e + 1, **{t: loss_weights(t, e, EPOCHS) for t in ("sst", "mnli")}}
                    for e in range(EPOCHS)
                ],
            },
            "checkpoint_selection": {
                "quality_tolerance_absolute": QUALITY_TOLERANCE_ABSOLUTE,
                "quality_criterion": "maximum mean validation F1",
                "efficiency_criterion": "minimum unweighted mean of per-task validation FLOPs / original expert FLOPs, subject to both F1 constraints",
                "tie_break": "higher mean validation F1 for equal efficiency cost",
                "validation_references": validation_references,
                "best_quality": best_quality_checkpoint,
                "best_efficiency": best_efficiency_checkpoint,
                "efficiency_status": "available" if best_efficiency_checkpoint is not None else "no_eligible_checkpoint_yet",
                "evaluation_preference": EVALUATION_CHECKPOINT,
            },
            "reference_validation_forward_flops": sum(
                r["flops_per_example"] * r["metrics"]["num_samples"] for r in validation_references.values()
            ),
            "reference_cost_note": "Two original-expert validation passes before training; reported separately from training/epoch validation FLOPs.",
            "early_stopping_enabled": False,

            "batch_size": int(BATCH_SIZE),
            "max_len": int(MAX_LEN),

            "tau_start": float(TAU_START),
            "tau_end": float(TAU_END),
            "tau_annealing_epochs": int(TAU_ANNEALING_EPOCHS),

            "efficient_eval": bool(EFFICIENT_EVAL),
            "show_batch_progress": bool(SHOW_BATCH_PROGRESS),

            "training_time_seconds": float(time.time() - start_time),
        }
        atomic_save_json(TRAINING_SUMMARY_PATH, summary)
        return summary


    training_history = []
    routing_history = []

    best_val_f1_avg = -1.0
    start_time = time.time()

    resume_contract = make_resume_contract()
    resume_best_states = restore_epoch_checkpoint(resume_contract)
    # The checkpoint is authoritative if interruption occurred between output-file writes.
    atomic_save_json(TRAINING_HISTORY_PATH, training_history)
    atomic_save_json(ROUTING_ANALYSIS_PATH, routing_history)
    training_summary = save_training_summary("training_in_progress" if training_history else "training_started")

    for epoch in range(len(training_history), EPOCHS):

        epoch_cost_by_task = {task: new_training_cost() for task in ("sst", "mnli")}

        tau = get_tau(
            epoch=epoch,
            tau_start=TAU_START,
            tau_end=TAU_END,
            annealing_epochs=TAU_ANNEALING_EPOCHS
        )

        train_metrics = train_one_epoch(
            model=routed_model,
            sst_loader=sst_train_loader,
            mnli_loader=mnli_train_loader,
            optimizer=optimizer,
            device=DEVICE,
            epoch=epoch,
            tau=tau,
            cost_by_task=epoch_cost_by_task
        )

        sst_val_metrics, sst_routing_stats = evaluate_task(
            model=routed_model,
            dataloader=sst_val_loader,
            task="sst",
            criterion=criterion,
            device=DEVICE,
            return_routing_stats=True,
            cost=epoch_cost_by_task["sst"]
        )

        mnli_val_metrics, mnli_routing_stats = evaluate_task(
            model=routed_model,
            dataloader=mnli_val_loader,
            task="mnli",
            criterion=criterion,
            device=DEVICE,
            return_routing_stats=True,
            cost=epoch_cost_by_task["mnli"]
        )

        val_loss_avg = (sst_val_metrics["classification_loss"] + mnli_val_metrics["classification_loss"]) / 2
        val_f1_avg = (sst_val_metrics["f1"] + mnli_val_metrics["f1"]) / 2
        val_acc_avg = (sst_val_metrics["accuracy"] + mnli_val_metrics["accuracy"]) / 2

        scheduler.step(val_loss_avg)

        if not all(math.isfinite(value) for value in
                   (val_loss_avg, sst_val_metrics["f1"], mnli_val_metrics["f1"])):
            raise RuntimeError("Non-finite validation metrics")
        is_best = val_f1_avg > best_val_f1_avg
        metrics_by_task = {"sst": sst_val_metrics, "mnli": mnli_val_metrics}
        efficiency = efficiency_candidate(epoch, metrics_by_task)
        old_efficiency = best_efficiency_checkpoint
        is_best_efficiency = efficiency["eligible"] and (
            old_efficiency is None
            or efficiency["mean_normalized_cost"] < old_efficiency["mean_normalized_cost"] - 1e-12
            or (
                abs(efficiency["mean_normalized_cost"] - old_efficiency["mean_normalized_cost"]) <= 1e-12
                and val_f1_avg > old_efficiency["mean_f1"]
            )
        )

        global_decision_counts = merge_routing_counts(
            sst_routing_stats,
            mnli_routing_stats,
            key_name="total_choice_counts"
        )

        global_example_counts = merge_routing_counts(
            sst_routing_stats,
            mnli_routing_stats,
            key_name="examples_with_choice"
        )

        routing_epoch_record = {
            "epoch": epoch + 1,
            "sst": sst_routing_stats,
            "mnli": mnli_routing_stats,
            "global_decision_counts": global_decision_counts,
            "global_example_counts": global_example_counts
        }

        routing_history.append(routing_epoch_record)

        if is_best:
            best_val_f1_avg = val_f1_avg
            best_epoch = epoch + 1
            best_routing_stats = routing_epoch_record
            best_validation_metrics = {"sst": sst_val_metrics, "mnli": mnli_val_metrics}
            resume_best_states["quality"] = mutable_model_state(routed_model)
            save_best_model(model=routed_model, save_path=ROUTED_MODEL_SAVE_PATH)
            best_quality_checkpoint = {
                "epoch": best_epoch, "mean_f1": float(val_f1_avg),
                "path": str(ROUTED_MODEL_SAVE_PATH),
                "validation_metrics": metrics_by_task, "routing_stats": routing_epoch_record,
            }

        if is_best_efficiency:
            resume_best_states["efficiency"] = mutable_model_state(routed_model)
            save_best_model(model=routed_model, save_path=BEST_EFFICIENCY_MODEL_PATH)
            best_efficiency_checkpoint = {
                **efficiency, "path": str(BEST_EFFICIENCY_MODEL_PATH),
                "routing_stats": routing_epoch_record,
            }

        current_lrs = get_optimizer_lrs(optimizer)

        epoch_record = {
            "epoch": epoch + 1,
            "tau": float(tau),
            "learning_rates": current_lrs,
            "efficient_eval": bool(EFFICIENT_EVAL),

            "loss_weights": {task: loss_weights(task, epoch, EPOCHS) for task in ("sst", "mnli")},
            "efficiency_selection": efficiency,
            "best_efficiency": bool(is_best_efficiency),

            "train": train_metrics,

            "validation": {
                "sst": sst_val_metrics,
                "mnli": mnli_val_metrics,
                "loss_avg": float(val_loss_avg),
                "accuracy_avg": float(val_acc_avg),
                "f1_avg": float(val_f1_avg)
            },

            "routing": routing_epoch_record,
            "best": bool(is_best),
            "training_cost": finish_routed_training_cost(epoch_cost_by_task)
        }

        training_history.append(epoch_record)
        for task in ("sst", "mnli"):
            merge_training_cost(training_cost_by_task[task], epoch_cost_by_task[task])

        save_epoch_checkpoint(resume_contract, resume_best_states)
        atomic_save_json(TRAINING_HISTORY_PATH, training_history)
        atomic_save_json(ROUTING_ANALYSIS_PATH, routing_history)

        training_summary = save_training_summary("training_in_progress")

        print_epoch_compact(
            epoch=epoch,
            epochs=EPOCHS,
            tau=tau,
            train_metrics=train_metrics,
            sst_val_metrics=sst_val_metrics,
            mnli_val_metrics=mnli_val_metrics,
            val_f1_avg=val_f1_avg,
            best=is_best
        )

        print("  Cost weights:", {t: loss_weights(t, epoch, EPOCHS)["compute_cost_weight"]
                                    for t in ("sst", "mnli")},
              "| entropy:", loss_weights("sst", epoch, EPOCHS)["entropy_bonus_weight"])
        print("  Efficiency eligible:", efficiency["eligible"],
              "| normalized cost:", efficiency["mean_normalized_cost"],
              "| new best efficiency:", bool(is_best_efficiency))
        # Complete all configured epochs, including the final-cost adaptation period.


    # ============================================================
    # SAVE TRAINING SUMMARY
    # ============================================================

    export_committed_best_models(resume_best_states)
    training_summary = save_training_summary("training_completed_evaluation_pending")

    print("\nTraining completed.")
    print(f"Best AVG Val F1: {best_val_f1_avg:.4f}")
    print("Best full model saved:", ROUTED_MODEL_SAVE_PATH)
    if best_efficiency_checkpoint is None:
        print("No admissible efficiency checkpoint: best-quality checkpoint remains available.")
    else:
        print("Best efficiency checkpoint:", BEST_EFFICIENCY_MODEL_PATH,
              "| epoch", best_efficiency_checkpoint["epoch"],
              "| mean normalized cost", best_efficiency_checkpoint["mean_normalized_cost"])


In [ ]:
def load_selected_checkpoint():
    global DEVICE, MODEL_PATH, Path, gc, name, routed_model, selected_checkpoint, selected_kind
    global torch
    # ============================================================
    # LOAD SELECTED CHECKPOINT: EFFICIENCY IF ELIGIBLE, OTHERWISE QUALITY
    # ============================================================

    import gc
    import torch
    from pathlib import Path

    if EVALUATION_CHECKPOINT == "quality":
        selected_kind = "quality"
    elif EVALUATION_CHECKPOINT == "efficiency_if_available":
        selected_kind = "efficiency" if best_efficiency_checkpoint is not None else "quality"
    else:
        raise ValueError("EVALUATION_CHECKPOINT must be quality or efficiency_if_available")

    selected_checkpoint = (
        best_efficiency_checkpoint if selected_kind == "efficiency" else best_quality_checkpoint
    )
    if selected_checkpoint is None:
        raise RuntimeError("No checkpoint was saved.")
    MODEL_PATH = Path(selected_checkpoint["path"])
    # Keep the existing report schema, but make its epoch/path refer to the evaluated checkpoint.
    training_summary["evaluated_checkpoint_kind"] = selected_kind
    training_summary["best_epoch"] = selected_checkpoint["epoch"]
    training_summary["best_model_path"] = str(MODEL_PATH)
    training_summary["best_val_f1_avg"] = selected_checkpoint["mean_f1"]
    training_summary["best_validation_metrics"] = selected_checkpoint["validation_metrics"]
    training_summary["final_routing_stats"] = selected_checkpoint["routing_stats"]
    training_summary["routing_stats_reference"] = "validation at the selected checkpoint epoch"
    print("Checkpoint selected for final evaluation:", selected_kind, "| epoch", selected_checkpoint["epoch"])
    if EVALUATION_CHECKPOINT == "efficiency_if_available" and best_efficiency_checkpoint is None:
        print("No checkpoint met both quality constraints with available FLOPs; using best quality.")

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for name in [
        "routed_model", "optimizer", "scheduler", "sst_backbone", "mnli_backbone",
        "merged_backbone", "shared_embeddings", "sst_layers", "mnli_layers", "merged_layers"
    ]:
        if name in globals():
            del globals()[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    routed_model = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
    routed_model.to(DEVICE)
    routed_model.eval()
    validate_merged_layout(routed_model)
    print("Selected routed model loaded:", MODEL_PATH)


In [ ]:
def evaluate_selected_checkpoint():
    global CHOICE_IDENTITY, CHOICE_NAMES, Counter, F, MANIFEST_PATH, NUM_CHOICES, NUM_LAYERS, Path
    global REPORT_JSON_PATH, ROUTING_TRACE_PATH, TAU_EVAL, accuracy_score, analysis, average_test, baseline_folder, baseline_report_paths
    global best_routing_analysis, comparison_manifest, compute_metrics, confusion_matrix, criterion, evaluate, evaluation_results, file_sha256
    global hashlib, json, key, loader, mnli_results, name, np, path
    global precision_recall_fscore_support, raw, record, report, result, routing_trace_row, snapshot, split
    global sst_results, task, texts, tokenizer_snapshot, torch
    # ============================================================
    # BEST-CHECKPOINT EVALUATION AND REPLAYABLE ROUTING TRACES
    # No training or alternative-path search is performed in this cell.
    # Validation/test traces use deterministic routing from the loaded best checkpoint.
    # ============================================================

    import hashlib
    import json
    from collections import Counter
    from pathlib import Path
    import numpy as np
    import torch
    import torch.nn.functional as F
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

    REPORT_JSON_PATH = RESULTS_PATH / f"routed_full_metrics_{RUN_TIMESTAMP}.json"
    ROUTING_TRACE_PATH = RESULTS_PATH / "routing_per_input"
    ROUTING_TRACE_PATH.mkdir(parents=True, exist_ok=True)
    MANIFEST_PATH = RESULTS_PATH / "comparison_manifest.json"
    criterion = torch.nn.CrossEntropyLoss()
    TAU_EVAL = globals().get("TAU_END", 0.1)
    CHOICE_NAMES = ["SST", "MNLI", "MERGED", "IDENTITY"]
    CHOICE_IDENTITY = 3
    NUM_LAYERS = 12
    NUM_CHOICES = 4


    def compute_metrics(y_true, y_pred, task):
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary" if task == "sst" else "macro", zero_division=0)
        return {"accuracy": float(accuracy_score(y_true, y_pred)),
                "precision": float(precision), "recall": float(recall), "f1": float(f1), "f1_average": "binary" if task == "sst" else "macro"}


    def file_sha256(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()


    def routing_trace_row(task, split, index, batch_index, text, input_ids,
                          attention_mask, token_type_ids, label, prediction,
                          logits, class_probabilities, classification_loss,
                          routing_choices, routing_logits, routing_probabilities):
        """One reproducible observation; cost is recorded at its measured batch scope."""
        if not (len(input_ids) == len(attention_mask) == len(token_type_ids)):
            raise ValueError("Token tensors have inconsistent lengths")
        if not (len(routing_choices) == len(routing_logits) == len(routing_probabilities) == NUM_LAYERS):
            raise ValueError("Expected one routing decision per layer")
        if any(c not in range(NUM_CHOICES) for c in routing_choices):
            raise ValueError("Unknown routing choice")
        if any(len(row) != NUM_CHOICES for row in routing_logits + routing_probabilities):
            raise ValueError("Expected four alternatives per routing decision")
        available_by_layer = ROUTING_AVAILABILITY["available_choices_by_layer"]
        for depth, (choice, probabilities) in enumerate(zip(routing_choices, routing_probabilities)):
            available = available_by_layer[depth]
            if CHOICE_NAMES[choice] not in available:
                raise ValueError("Trace selected an unavailable encoder choice")
            if any(probability != 0.0 for name, probability in zip(CHOICE_NAMES, probabilities)
                   if name not in available):
                raise ValueError("Unavailable encoder choices must have zero probability")
        # A masked logit is -inf internally, but JSON must remain standards-compliant.
        # Preserve the four-column schema: null identifies an unavailable alternative.
        exported_routing_logits = [
            [None if CHOICE_NAMES[choice] not in available_by_layer[depth] else value
             for choice, value in enumerate(row)]
            for depth, row in enumerate(routing_logits)
        ]
        active = sum(c != CHOICE_IDENTITY for c in routing_choices)
        # Hash unpadded tokens, independently of the split padding length.
        unpadded = [(i, t) for i, t, m in zip(input_ids, token_type_ids, attention_mask) if m]
        fingerprint = hashlib.sha256(json.dumps(
            unpadded, separators=(",", ":")).encode("utf-8")).hexdigest()
        return {
            "example_id": f"{task}:{split}:{index}",
            "run_example_index": index, "task": task, "split": split,
            "input_sha256": fingerprint, "text": str(text),
            "input_ids": input_ids, "attention_mask": attention_mask,
            "token_type_ids": token_type_ids, "padded_sequence_length": len(input_ids),
            "non_padding_tokens": int(sum(attention_mask)),
            "label": label, "prediction": prediction, "correct": label == prediction,
            "logits": logits, "class_probabilities": class_probabilities,
            "classification_loss": classification_loss,
            "route": [CHOICE_NAMES[c] for c in routing_choices],
            "routing_choice_indices": routing_choices,
            "routing_logits": exported_routing_logits, "routing_probabilities": routing_probabilities,
            "available_choices_by_layer": ROUTING_AVAILABILITY["available_choices_by_layer"],
            "unavailable_logit_encoding": "null",
            "active_layers": active, "skipped_layers": NUM_LAYERS - active,
            "flops_batch_index": batch_index,
        }


    @torch.no_grad()
    def evaluate(model, dataloader, task, split, texts):
        """One efficient pass for predictions, routes and executed FLOPs together.

        Requires the existing non-shuffled validation/test loader. Raw rows and batch
        costs are streamed to disk; no hidden-state histories or graphs are retained.
        """
        if split not in ("validation", "test"):
            raise ValueError("Only validation and test are exported here")
        if not isinstance(dataloader.sampler, torch.utils.data.SequentialSampler):
            raise ValueError("Trace export requires a sequential sampler")
        if len(texts) != len(dataloader.dataset):
            raise ValueError("Texts and evaluation dataset are not aligned")
        if not texts:
            raise ValueError("Cannot export an empty split")
        model.eval()
        inference_cost = new_inference_cost()
        total_loss, total_samples = 0.0, 0
        y_true, y_pred = [], []
        choice_counts = np.zeros((NUM_LAYERS, NUM_CHOICES), dtype=np.int64)
        route_counts, active_histogram = Counter(), Counter()
        trace_path = ROUTING_TRACE_PATH / f"{task}_{split}.jsonl"
        batch_cost_path = ROUTING_TRACE_PATH / f"{task}_{split}_batch_flops.jsonl"
        trace_tmp = trace_path.with_suffix(".jsonl.partial")
        cost_tmp = batch_cost_path.with_suffix(".jsonl.partial")
        with trace_tmp.open("w", encoding="utf-8") as trace_file, cost_tmp.open("w", encoding="utf-8") as cost_file:
            for batch_index, cpu_batch in enumerate(dataloader):
                cpu_ids, cpu_mask, cpu_types, cpu_labels = [t.tolist() for t in cpu_batch]
                input_ids, attention_mask, token_type_ids, labels = [t.to(DEVICE) for t in cpu_batch]
                with RoutedFlopCounter() as counter:
                    outputs = model(
                        input_ids=input_ids, attention_mask=attention_mask,
                        token_type_ids=token_type_ids, task=task, tau=TAU_EVAL,
                        efficient_inference=True, return_routing=True)
                # Metrics/serialization are outside the model FLOP counter.
                add_inference_cost(inference_cost, counter, "routed_model", input_ids)
                logits = outputs["logits"]
                losses = F.cross_entropy(logits, labels, reduction="none").cpu().tolist()
                predictions = logits.argmax(dim=1).cpu().tolist()
                class_probs = torch.softmax(logits, dim=1).cpu().tolist()
                logits_cpu = logits.cpu().tolist()
                choices = outputs["routing_choices"].argmax(dim=-1).cpu().tolist()
                routing_logits = outputs["routing_logits"].cpu().tolist()
                routing_probs = outputs["routing_probs"].cpu().tolist()
                batch_size = len(cpu_labels)
                counted = int(counter.get_total_flops())
                uncovered = sorted(str(op) for op in counter.uncovered_ops)
                complete = not uncovered and counted > 0
                batch_record = {
                    "task": task, "split": split, "batch_index": batch_index,
                    "first_run_example_index": total_samples, "num_examples": batch_size,
                    "padded_sequence_length": int(input_ids.shape[1]),
                    "total_flops": counted if complete else None,
                    "counted_flops": counted,
                    "coverage": "complete_for_convention" if complete else "incomplete",
                    "uncovered_operators": uncovered,
                }
                cost_file.write(json.dumps(batch_record, allow_nan=False) + "\n")
                for i in range(batch_size):
                    row = routing_trace_row(
                        task, split, total_samples + i, batch_index, texts[total_samples + i],
                        cpu_ids[i], cpu_mask[i], cpu_types[i], cpu_labels[i], predictions[i],
                        logits_cpu[i], class_probs[i], losses[i], choices[i],
                        routing_logits[i], routing_probs[i])
                    trace_file.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
                    route_counts[tuple(choices[i])] += 1
                    active_histogram[row["active_layers"]] += 1
                    for depth, choice in enumerate(choices[i]):
                        choice_counts[depth, choice] += 1
                total_loss += sum(losses)
                y_true.extend(cpu_labels)
                y_pred.extend(predictions)
                total_samples += batch_size
                del outputs, logits
        if total_samples != len(texts):
            raise ValueError("Incomplete export; partial files have been retained")
        trace_tmp.replace(trace_path)
        cost_tmp.replace(batch_cost_path)
        metrics = compute_metrics(y_true, y_pred, task)
        metrics.update(loss=float(total_loss / total_samples),
                       classification_loss=float(total_loss / total_samples), num_samples=total_samples)
        total_counts = choice_counts.sum(axis=0)
        avg_active = sum(k * v for k, v in active_histogram.items()) / total_samples
        routing = {
            "num_examples": total_samples, "num_layers": NUM_LAYERS,
            "routing_availability": ROUTING_AVAILABILITY,
            "total_routing_decisions": total_samples * NUM_LAYERS,
            "average_active_non_identity_layers_per_example": avg_active,
            "average_skipped_identity_layers_per_example": NUM_LAYERS - avg_active,
            "total_choice_counts": dict(zip(CHOICE_NAMES, total_counts.tolist())),
            "total_choice_percentages": dict(zip(CHOICE_NAMES, (100 * total_counts / (total_samples * NUM_LAYERS)).tolist())),
            "per_layer": [{
                "layer": depth + 1,
                "available_choices": ROUTING_AVAILABILITY["available_choices_by_layer"][depth],
                "hard_choice_percentages": dict(zip(CHOICE_NAMES, (100 * row / total_samples).tolist())),
                "most_selected_choice": CHOICE_NAMES[int(row.argmax())],
            } for depth, row in enumerate(choice_counts)],
            "num_unique_routes": len(route_counts),
            "active_layer_histogram": dict(sorted(active_histogram.items())),
        }
        cost_result = finish_inference_cost(inference_cost, total_samples)
        cost_result["scope"] = f"{split} inference: inputs to logits, including router; metrics and export excluded"
        return {
            "metrics": metrics,
            "confusion_matrix": confusion_matrix(y_true, y_pred, labels=list(range(2 if task == "sst" else 3))).tolist(),
            "inference_cost": cost_result, "routing": routing,
            "per_input": {
                "schema_version": 2, "num_examples": total_samples,
                "unavailable_logit_encoding": "null",
                "jsonl_path": str(trace_path), "sha256": file_sha256(trace_path),
                "batch_flops_jsonl_path": str(batch_cost_path),
                "batch_flops_sha256": file_sha256(batch_cost_path),
            },
        }


    # Evaluates the loaded best checkpoint, never the final unsaved training state.
    # Preserve a manifest with completion status in case evaluation is interrupted.
    comparison_manifest = {
        "schema_version": 1, "measurement_status": "pending", "status": "evaluation_in_progress", "run_timestamp": RUN_TIMESTAMP,
        "checkpoint": {"path": str(MODEL_PATH), "sha256": file_sha256(MODEL_PATH),
                       "best_epoch": training_summary["best_epoch"],
                       "selection_kind": selected_kind,
                       "selection_criteria": training_summary["checkpoint_selection"],
                       "format": "torch.save full RoutedBertModel; SST/MNLI full encoders, only merged encoder layer 1, router, embeddings, pooler and heads",
                       "class_definitions": ["ClassifierHead", "SharedMLPRouter", "RoutedBertModel"],
                       "architecture": str(routed_model)},
        "report_path": str(REPORT_JSON_PATH),
        "criteria": {"quality_tolerance_absolute": QUALITY_TOLERANCE_ABSOLUTE,
                     "minimum_flops_saving_fraction": MIN_INFERENCE_FLOPS_SAVING,
                     "flops_comparison": "strictly_greater_than",
                     "quality_comparison": "routed_score >= baseline_score - tolerance; no upper limit",
                     "primary_metric": {"sst": "binary_f1", "mnli": "macro_f1"},
                     "secondary_metric": "accuracy", "uncertainty_analysis": False},
        "inference_mode": "eval, no_grad, deterministic argmax, efficient selected-layer execution",
        "index_convention": "zero-based position in the run split; not the original parquet index; input_sha256 identifies unpadded tokens",
        "cost_scope": "FLOPs measured per executed batch and aggregated per split; no invented per-input allocation",
        "additional_evaluation_cost_scope": "validation replay is diagnostic evaluation, excluded from training totals; test is the standard test evaluation",
        "comparisons_not_yet_executed": [
            "same skip positions with only SST/MNLI/merged expert layers",
            "alternative paths changing experts or skips"],
        "routing_availability": ROUTING_AVAILABILITY,
        "experiment": experiment_config, "splits": {},
    }
    atomic_save_json(MANIFEST_PATH, comparison_manifest)
    evaluation_results = {}
    for task, split, loader, texts in (
        ("sst", "validation", sst_val_loader, x_sst_val_run),
        ("mnli", "validation", mnli_val_loader, x_mnli_val_run),
        ("sst", "test", sst_test_loader, x_sst_test_run),
        ("mnli", "test", mnli_test_loader, x_mnli_test_run),
    ):
        key = f"{task}_{split}"
        result = evaluate(routed_model, loader, task, split, texts)
        evaluation_results[key] = result
        comparison_manifest["splits"][key] = result
        atomic_save_json(MANIFEST_PATH, comparison_manifest)
        print(f"{key}: {result['metrics']['num_samples']} esempi; tracce in {result['per_input']['jsonl_path']}")

    sst_results = evaluation_results["sst_test"]
    mnli_results = evaluation_results["mnli_test"]
    average_test = {key: float((sst_results["metrics"][key] + mnli_results["metrics"][key]) / 2)
                    for key in ("loss", "classification_loss", "accuracy", "precision", "recall", "f1")}
    best_routing_analysis = {f"{task}_validation": evaluation_results[f"{task}_validation"]["routing"]
                             for task in ("sst", "mnli")}

    # Baseline reports are copied as evidence, not recomputed or automatically paired to this run.
    baseline_report_paths = {
        "sst_expert": BASELINES_PATH / "EvaluationExpertSST/Results/expert_a_metrics.json",
        "mnli_expert": BASELINES_PATH / "EvaluationExpertMNLI/Results/expert_b_metrics.json",
        "multitask": BASELINES_PATH / "EvaluationExpertMultitaskBert/Results/multitask_bert_test_results.json",
        "weight_averaging": BASELINES_PATH / "EvaluationExpertWeigthAveraging/Results/weight_averaging_test_results.json",
    }
    baseline_folder = RESULTS_PATH / "baseline_reports"
    baseline_folder.mkdir(parents=True, exist_ok=True)
    comparison_manifest["baseline_reports"] = {}
    for name, path in baseline_report_paths.items():
        record = {"source_path": str(path), "status": "missing"}
        if path.is_file():
            raw = path.read_bytes()
            json.loads(raw)  # fail explicitly for a malformed report
            snapshot = baseline_folder / f"{name}.json"
            snapshot.write_bytes(raw)
            record.update(status="available", snapshot_path=str(snapshot),
                          sha256=hashlib.sha256(raw).hexdigest())
        else:
            print(f"Report baseline assente: {path}; il confronto resta da completare.")
        comparison_manifest["baseline_reports"][name] = record

    tokenizer_snapshot = RESULTS_PATH / "tokenizer"
    tokenizer.save_pretrained(tokenizer_snapshot)
    comparison_manifest["tokenizer_path"] = str(tokenizer_snapshot)
    comparison_manifest["expert_directories"] = {"sst": str(SST_EXPERT_PATH), "mnli": str(MNLI_EXPERT_PATH)}
    comparison_manifest["status"] = "evaluation_completed"
    atomic_save_json(MANIFEST_PATH, comparison_manifest)

    training_summary["status"] = "evaluation_completed"
    training_summary["updated_at"] = datetime.now().astimezone().isoformat(timespec="seconds")
    training_summary["report_path"] = str(REPORT_JSON_PATH)
    training_summary["comparison_manifest_path"] = str(MANIFEST_PATH)
    atomic_save_json(TRAINING_SUMMARY_PATH, training_summary)
    report = {
        "experiment": {**experiment_config,
            "report_created_at": datetime.now().astimezone().isoformat(timespec="seconds"),
            "best_epoch": training_summary["best_epoch"], "checkpoint_used": str(MODEL_PATH),
            "checkpoint_selection_kind": selected_kind,
            "checkpoint_sha256": comparison_manifest["checkpoint"]["sha256"],
            "test_efficient_inference": True, "test_tau": float(TAU_EVAL)},
        "detailed_results": {
            "training_summary": training_summary, "training_history": training_history,
            "routing_history": routing_history, "average_metrics_by_split": {"test": average_test},
            "routing_analysis_required": best_routing_analysis,
            "splits": evaluation_results, "training_cost": training_summary["training_cost"],
            "mnli_representation_compatibility": {"status": "pending"},
            "routing_per_input": {key: result["per_input"] for key, result in evaluation_results.items()},
        },
        "comparison_manifest_path": str(MANIFEST_PATH),
        "summary": {
            "best_epoch": training_summary["best_epoch"],
            "training_flops": training_summary["training_cost"]["training_flops"],
            "validation_flops": training_summary["training_cost"]["validation_flops"],
            "preparation_flops": merge_cost["total_flops"],
            "test": {task: {
                "loss": result["metrics"]["loss"], "accuracy": result["metrics"]["accuracy"],
                "f1": result["metrics"]["f1"],
                "inference_flops_per_example": result["inference_cost"]["flops_per_example"],
                "inference_total_flops": result["inference_cost"]["total_flops"],
                "inference_flops_coverage": result["inference_cost"]["coverage"]["status"],
            } for task, result in (("sst", sst_results), ("mnli", mnli_results))},
            "routing_on_validation": {task: {
                "average_active_non_identity_layers_per_example": analysis["average_active_non_identity_layers_per_example"],
                "total_choice_percentages": analysis["total_choice_percentages"],
            } for task, analysis in (("sst", best_routing_analysis["sst_validation"]),
                                    ("mnli", best_routing_analysis["mnli_validation"]))},
        },
    }
    atomic_save_json(REPORT_JSON_PATH, report)
    print("Report salvato:", REPORT_JSON_PATH)
    print("Dati per il notebook di confronto:", MANIFEST_PATH)
    print("I confronti con percorsi alternativi richiedono ancora nuove inferenze, non nuovo training.")


In [ ]:
def print_run_report():
    global CHOICE_LABELS, CHOICE_ORDER, Path, SPLIT_TO_PRINT, format_flops, json, load_json, print_clean_report
    global print_computational_cost, print_confusion_matrix, print_performance_analysis, print_routing_analysis, print_routing_analysis_for_task, print_section, print_subsection, print_task_performance
    # ============================================================
    # CLEAN FINAL REPORT FROM SAVED JSON
    # Routed Model
    # 1. Experiment description
    # 2. Detailed training/evaluation and computational cost
    # 3. Compact summary
    # ============================================================

    from pathlib import Path
    import json

    # ============================================================
    # PATHS
    # ============================================================

    # RESULTS_PATH is the run-specific directory configured in the first cell.

    # REPORT_JSON_PATH is the timestamped report saved by the evaluation cell.

    SPLIT_TO_PRINT = "test"

    CHOICE_ORDER = ["SST", "MNLI", "MERGED", "IDENTITY"]

    CHOICE_LABELS = {
        "SST": "A/SST",
        "MNLI": "B/MNLI",
        "MERGED": "M",
        "IDENTITY": "Id"
    }


    # ============================================================
    # LOAD JSON
    # ============================================================

    def load_json(path):
        path = Path(path)

        if not path.exists():
            raise FileNotFoundError(f"JSON file not found: {path}")

        with open(path, "r") as f:
            return json.load(f)


    # ============================================================
    # PRINT HELPERS
    # ============================================================

    def print_section(title):
        print("\n" + "=" * 90)
        print(title)
        print("=" * 90)


    def print_subsection(title):
        print("\n" + title)
        print("-" * len(title))


    # ============================================================
    # ROUTING ANALYSIS
    # ============================================================

    def print_routing_analysis_for_task(task_title, analysis):
        print_subsection(task_title)

        print(
            f"Average active non-identity layers per example: "
            f"{analysis['average_active_non_identity_layers_per_example']:.4f} / {analysis['num_layers']}"
        )

        print(
            f"Average skipped identity layers per example: "
            f"{analysis['average_skipped_identity_layers_per_example']:.4f} / {analysis['num_layers']}"
        )

        print("\nOverall router choices:")
        for choice in CHOICE_ORDER:
            label = CHOICE_LABELS[choice]
            value = analysis["total_choice_percentages"][choice]
            print(f"{label:<10}: {value:>6.2f}%")

        print("\nFraction of choices at each layer depth:")
        print(
            f"{'Layer':<8}"
            f"{'A/SST %':>10}"
            f"{'B/MNLI %':>12}"
            f"{'M %':>10}"
            f"{'Id %':>10}"
            f"{'Most selected':>18}"
        )
        print("-" * 80)

        for row in analysis["per_layer"]:
            pct = row["hard_choice_percentages"]

            print(
                f"{row['layer']:<8}"
                f"{pct['SST']:>10.2f}"
                f"{pct['MNLI']:>12.2f}"
                f"{pct['MERGED']:>10.2f}"
                f"{pct['IDENTITY']:>10.2f}"
                f"{CHOICE_LABELS[row['most_selected_choice']]:>18}"
            )


    def print_routing_analysis(report):
        print_section("ROUTING ANALYSIS - VALIDATION SETS")

        print(
            "Router choices are printed as:\n"
            "A/SST = SST expert layer\n"
            "B/MNLI = MNLI expert layer\n"
            "M = merged layer (available only at layer 1; unavailable at layers 2-12)\n"
            "Id = identity / skipped layer"
        )

        routing = report["routing_analysis_required"]

        print_routing_analysis_for_task(
            "SST VALIDATION INPUTS",
            routing["sst_validation"]
        )

        print_routing_analysis_for_task(
            "MNLI VALIDATION INPUTS",
            routing["mnli_validation"]
        )


    # ============================================================
    # PERFORMANCE ANALYSIS
    # ============================================================

    def print_confusion_matrix(cm):
        print("\nConfusion matrix")
        print("Rows = true class, columns = predicted class\n")

        num_cols = len(cm[0])

        header = " " * 10
        for j in range(num_cols):
            header += f"Pred {j:<10}"
        print(header)

        for i, row in enumerate(cm):
            line = f"True {i:<5}"
            for value in row:
                line += f"{value:<16}"
            print(line)


    def print_task_performance(report, task, split):
        key = f"{task}_{split}"

        result = report["splits"][key]
        metrics = result["metrics"]

        print_subsection(f"{task.upper()} - {split.upper()} SET")

        loss = metrics.get("loss", metrics.get("classification_loss"))

        print(f"Loss:      {loss:.4f}")
        print(f"Accuracy:  {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall:    {metrics['recall']:.4f}")
        print(f"F1:        {metrics['f1']:.4f}")
        print(f"Samples:   {metrics['num_samples']}")

        print_confusion_matrix(result["confusion_matrix"])


    def print_performance_analysis(report, split):
        print_section(f"PERFORMANCE ANALYSIS - {split.upper()} SET")

        print_task_performance(report, "sst", split)
        print_task_performance(report, "mnli", split)


    # ============================================================
    # RUN REPORT
    # ============================================================

    def format_flops(value):
        return "INCOMPLETE (see coverage)" if value is None else f"{value:,.0f}"


    def print_computational_cost(details):
        cost = details["training_cost"]
        print_subsection("TRAINING COST - ALL COMPLETED EPOCHS")
        print("Training FLOPs:", format_flops(cost["training_flops"]))
        print("Validation FLOPs:", format_flops(cost["validation_flops"]))
        print("One-time weight preparation FLOPs:", format_flops(cost["preparation_cost"]["total_flops"]))
        print("Total including preparation:", format_flops(cost["total_with_preparation_flops"]))
        print("Coverage:", cost["coverage"]["status"])
        if cost["coverage"]["uncovered_operators"]:
            print("Uncovered operators:", ", ".join(cost["coverage"]["uncovered_operators"]))
        print("\n" + json.dumps(cost["convention"], indent=2))
        for task, task_cost in cost["by_task"].items():
            print_subsection(f"{task.upper()} - TRAINING / VALIDATION COMPONENTS")
            for component, value in task_cost["components"].items():
                print(f"{component:<32} {format_flops(value['total_flops']):>25}")
        print_subsection("FLOPs BY EPOCH")
        print(f"{'Epoch':<8} {'Training FLOPs':>25} {'Validation FLOPs':>25}")
        for row in cost["per_epoch"]:
            print(f"{row['epoch']:<8} {format_flops(row['training_flops']):>25} {format_flops(row['validation_flops']):>25}")
        print_subsection("TEST INFERENCE COST - BEST CHECKPOINT")
        for task in ("sst", "mnli"):
            item = details["splits"][f"{task}_test"]["inference_cost"]
            print(f"{task.upper()}: FLOPs/example={format_flops(item['flops_per_example'])}; total={format_flops(item['total_flops'])}; N={item['num_examples']}")
            print("  Sequence lengths:", item["input"]["observed_sequence_lengths"])
            print("  Coverage:", item["coverage"]["status"])
            if item["coverage"]["uncovered_operators"]:
                print("  Uncovered:", ", ".join(item["coverage"]["uncovered_operators"]))


    def print_clean_report(json_path, split):
        report = load_json(json_path)
        details = report["detailed_results"]
        print_section("1. EXPERIMENT DESCRIPTION")
        print("Report:", json_path)
        experiment = report["experiment"]
        print("Run timestamp:", experiment["run_started_at"])
        print("Checkpoint:", experiment["checkpoint_used"])
        print("Best epoch:", experiment["best_epoch"])
        print_subsection("CONFIGURED PARAMETERS")
        for key, value in experiment["parameters"].items():
            print(f"{key:<36}: {value}")
        for key in ("optimizer", "scheduler", "input_preparation"):
            print_subsection(key.upper())
            print(json.dumps(experiment[key], indent=2))
        print("Trainable parameters:", experiment["trainable_params"])
        print("Total parameters:", experiment["total_params"])
        print("Device / dtype:", experiment["device"], experiment["dtype"])
        print("PyTorch / Transformers:", experiment["torch_version"], experiment["transformers_version"])
        print("Full backbone configurations and architecture are included in the saved JSON.")

        print_section("2. DETAILED RESULTS")
        print_subsection("TRAINING HISTORY")
        print(f"{'Epoch':<6} {'Task':<6} {'Train loss':>12} {'Train acc':>11} {'Train F1':>10} {'Val loss':>11} {'Val acc':>10} {'Val F1':>10} {'Best':>6}")
        for row in details["training_history"]:
            for task in ("sst", "mnli"):
                tr, va = row["train"][task], row["validation"][task]
                print(f"{row['epoch']:<6} {task:<6} {tr['loss']:>12.4f} {tr['accuracy']:>11.4f} {tr['f1']:>10.4f} {va['loss']:>11.4f} {va['accuracy']:>10.4f} {va['f1']:>10.4f} {str(row['best']):>6}")
        print_performance_analysis(details, split)
        print_routing_analysis(details)
        print_computational_cost(details)

        print_section("3. SUMMARY - BEST CHECKPOINT")
        print("Best epoch:", report["summary"]["best_epoch"])
        print(f"{'Task':<8} {'Test loss':>12} {'Accuracy':>12} {'F1':>12} {'Inference FLOPs/example':>28}")
        for task, row in report["summary"]["test"].items():
            print(f"{task.upper():<8} {row['loss']:>12.4f} {row['accuracy']:>12.4f} {row['f1']:>12.4f} {format_flops(row['inference_flops_per_example']):>28}")
            routing = report["summary"]["routing_on_validation"][task]
            print("  Validation active layers:", routing["average_active_non_identity_layers_per_example"], "/", NUM_LAYERS)
            print("  Validation choices (%):", routing["total_choice_percentages"])
        print("Training FLOPs:", format_flops(report["summary"]["training_flops"]))
        print("Validation FLOPs:", format_flops(report["summary"]["validation_flops"]))


    print_clean_report(json_path=REPORT_JSON_PATH, split=SPLIT_TO_PRINT)


In [ ]:
def check_mnli_representations():
    global F, MAX_BATCHES, all_correct, all_final_cos, all_final_l2, all_losses, all_merged_counts, all_switches
    global attention_mask, batch, batch_idx, batch_size, ce_none, choice_idx, choices, choices_per_layer
    global cls_final, cls_hidden, cls_mnli, cls_routed, compatibility_analysis, correct, cos, extended_attention_mask
    global extract_hidden, f, final_cos, final_l2, finite_correlation, hard_choice, hidden_mnli_only, hidden_routed
    global i, input_ids, l2, labels, layer_cos_sums, layer_idx, layer_l2_sums, logits
    global losses, mask, merged_counts, name, np, preds, q1, q2
    global report, router_logits, soft_probs, summarize_group, switches, token_type_ids, torch, total_examples
    # ============================================================
    # MNLI REPRESENTATION COMPATIBILITY CHECK
    # Routed path vs MNLI-only path
    # ============================================================

    import torch
    import torch.nn.functional as F
    import numpy as np

    MAX_BATCHES = None   # None = tutta la validation MNLI, oppure metti 20 per test rapido

    routed_model.eval()
    routed_model.to(DEVICE)

    ce_none = torch.nn.CrossEntropyLoss(reduction="none")


    def extract_hidden(layer_output):
        if torch.is_tensor(layer_output):
            return layer_output
        return layer_output[0]


    def summarize_group(mask, losses, correct):
        count = int(mask.sum())
        return {
            "num_examples": count,
            "accuracy": float(correct[mask].mean()) if count else None,
            "loss": float(losses[mask].mean()) if count else None,
        }


    all_losses = []
    all_correct = []
    all_final_cos = []
    all_final_l2 = []
    all_switches = []
    all_merged_counts = []

    layer_cos_sums = np.zeros(12)
    layer_l2_sums = np.zeros(12)
    total_examples = 0

    with torch.no_grad():
        for batch_idx, batch in enumerate(mnli_val_loader):

            if MAX_BATCHES is not None and batch_idx >= MAX_BATCHES:
                break

            input_ids, attention_mask, token_type_ids, labels = batch

            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            token_type_ids = token_type_ids.to(DEVICE)
            labels = labels.to(DEVICE)

            batch_size = input_ids.size(0)

            hidden_routed = routed_model.shared_embeddings(
                input_ids=input_ids,
                token_type_ids=token_type_ids
            )

            hidden_mnli_only = hidden_routed.clone()

            extended_attention_mask = routed_model.reference_backbone.get_extended_attention_mask(
                attention_mask,
                input_ids.size(),
                dtype=hidden_routed.dtype
            )

            choices_per_layer = []

            for layer_idx in range(routed_model.num_layers):

                # MNLI-only path
                hidden_mnli_only = extract_hidden(
                    routed_model.mnli_layers[layer_idx](
                        hidden_mnli_only,
                        attention_mask=extended_attention_mask
                    )
                )

                # Routed path
                cls_hidden = hidden_routed[:, 0, :]

                router_logits, soft_probs, hard_choice = routed_model.router.forward_step(
                    cls_hidden=cls_hidden,
                    layer_idx=layer_idx,
                    tau=0.1
                )

                hidden_routed = routed_model.apply_wasteful_layer(
                    hidden_states=hidden_routed,
                    extended_attention_mask=extended_attention_mask,
                    layer_idx=layer_idx,
                    hard_choice=hard_choice
                )

                choice_idx = hard_choice.argmax(dim=1)
                choices_per_layer.append(choice_idx)

                # Layer-wise distance
                cls_routed = hidden_routed[:, 0, :]
                cls_mnli = hidden_mnli_only[:, 0, :]

                cos = F.cosine_similarity(cls_routed, cls_mnli, dim=1)
                l2 = torch.norm(cls_routed - cls_mnli, dim=1)

                layer_cos_sums[layer_idx] += cos.sum().item()
                layer_l2_sums[layer_idx] += l2.sum().item()

            # Final prediction from routed path
            if routed_model.pooler is not None:
                cls_final = routed_model.pooler(hidden_routed)
            else:
                cls_final = hidden_routed[:, 0, :]

            logits = routed_model.mnli_head(cls_final)
            losses = ce_none(logits, labels)
            preds = torch.argmax(logits, dim=1)
            correct = (preds == labels).float()

            choices = torch.stack(choices_per_layer, dim=1)

            switches = (choices[:, 1:] != choices[:, :-1]).sum(dim=1)
            merged_counts = (choices == 2).sum(dim=1)

            final_cos = F.cosine_similarity(
                hidden_routed[:, 0, :],
                hidden_mnli_only[:, 0, :],
                dim=1
            )

            final_l2 = torch.norm(
                hidden_routed[:, 0, :] - hidden_mnli_only[:, 0, :],
                dim=1
            )

            all_losses.extend(losses.cpu().numpy())
            all_correct.extend(correct.cpu().numpy())
            all_final_cos.extend(final_cos.cpu().numpy())
            all_final_l2.extend(final_l2.cpu().numpy())
            all_switches.extend(switches.cpu().numpy())
            all_merged_counts.extend(merged_counts.cpu().numpy())

            total_examples += batch_size


    all_losses = np.array(all_losses)
    all_correct = np.array(all_correct)
    all_final_cos = np.array(all_final_cos)
    all_final_l2 = np.array(all_final_l2)
    all_switches = np.array(all_switches)
    all_merged_counts = np.array(all_merged_counts)

    def finite_correlation(x, y):
        if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0:
            return None
        value = float(np.corrcoef(x, y)[0, 1])
        return value if np.isfinite(value) else None


    if total_examples:
        q1, q2 = np.quantile(all_final_l2, [1/3, 2/3])
        compatibility_analysis = {
            "task": "mnli", "split": "validation", "best_epoch": training_summary["best_epoch"],
            "num_examples": int(total_examples), "max_batches": MAX_BATCHES,
            "scope": "diagnostic comparison with MNLI-only layers using the same merged embeddings/pooler/head; excluded from train/inference FLOPs",
            "layer_distances": [
                {"layer": i + 1, "mean_cosine": float(layer_cos_sums[i] / total_examples),
                 "mean_l2": float(layer_l2_sums[i] / total_examples)} for i in range(NUM_LAYERS)
            ],
            "cosine_loss_correlation": finite_correlation(all_final_cos, all_losses),
            "l2_loss_correlation": finite_correlation(all_final_l2, all_losses),
            "distance_tertile_boundaries": [float(q1), float(q2)],
            "by_distance": {
                name: summarize_group(mask, all_losses, all_correct) for name, mask in (
                    ("low", all_final_l2 <= q1),
                    ("mid", (all_final_l2 > q1) & (all_final_l2 <= q2)),
                    ("high", all_final_l2 > q2),
                )
            },
            "by_switches": {
                name: summarize_group(mask, all_losses, all_correct) for name, mask in (
                    ("0-1", all_switches <= 1),
                    ("2-3", (all_switches >= 2) & (all_switches <= 3)),
                    ("4+", all_switches >= 4),
                )
            },
            "routing_availability": ROUTING_AVAILABILITY,
            "by_merged_layers": {
                "0": summarize_group(all_merged_counts == 0, all_losses, all_correct),
                "1": summarize_group(all_merged_counts == 1, all_losses, all_correct),
            },
        }
    else:
        compatibility_analysis = {"task": "mnli", "split": "validation", "num_examples": 0}

    with open(REPORT_JSON_PATH, "r", encoding="utf-8") as f:
        report = json.load(f)
    compatibility_analysis["status"] = "completed"
    report["detailed_results"]["mnli_representation_compatibility"] = compatibility_analysis
    atomic_save_json(REPORT_JSON_PATH, report)
    print("MNLI diagnostic analysis saved in:", REPORT_JSON_PATH)


In [ ]:
RUN_STATE_NAMES = ['AdamW', 'AutoModel', 'AutoTokenizer', 'BASELINES_PATH', 'BATCH_SIZE', 'BEST_EFFICIENCY_MODEL_PATH', 'CHOICE_IDENTITY', 'CHOICE_LABELS', 'CHOICE_MERGED', 'CHOICE_MNLI', 'CHOICE_NAMES', 'CHOICE_ORDER', 'CHOICE_SST', 'CLASSIFICATION_LOSS_WEIGHT', 'ClassifierHead', 'Counter', 'DATASET_PATH', 'DEBUG_FRACTION', 'DEPTH_EMBEDDING_SIZE', 'DEVICE', 'DROPOUT_PROBABILITY', 'DataLoader', 'EARLY_STOPPING_ENABLED', 'EARLY_STOPPING_MIN_DELTA', 'EARLY_STOPPING_PATIENCE', 'EFFICIENT_EVAL', 'ENTROPY_BONUS_WEIGHT_END', 'ENTROPY_BONUS_WEIGHT_START', 'ENTROPY_RAMP_END_FRACTION', 'EPOCHS', 'EVALUATION_CHECKPOINT', 'EXPERIMENT_MODELS_PATH', 'EXPERIMENT_NAME', 'EXPERIMENT_RESULTS_PATH', 'EarlyStopping', 'F', 'FlopCounterMode', 'GRADIENT_CLIPPING', 'HEAD_HIDDEN_SIZE', 'HEAD_LEARNING_RATE', 'HEAD_WEIGHT_DECAY', 'HIDDEN_SIZE', 'InferenceFlopCounter', 'LEARNING_RATE', 'LR_FACTOR', 'LR_PATIENCE', 'MANIFEST_PATH', 'MAX_BATCHES', 'MAX_GRAD_NORM', 'MAX_LEN', 'MERGED_MODE', 'MERGE_ALPHA_SST', 'MERGE_BETA_MNLI', 'MIN_HEAD_LR', 'MIN_INFERENCE_FLOPS_SAVING', 'MIN_LR', 'MIN_ROUTER_LR', 'MNLI_BACKBONE_PATH', 'MNLI_COMPUTE_COST_WEIGHT_END', 'MNLI_COMPUTE_COST_WEIGHT_START', 'MNLI_COST_RAMP_END_FRACTION', 'MNLI_DATASET_PATH', 'MNLI_EXPERT_PATH', 'MNLI_X_TEST_PATH', 'MNLI_X_TRAIN_PATH', 'MNLI_X_VAL_PATH', 'MNLI_Y_TEST_PATH', 'MNLI_Y_TRAIN_PATH', 'MNLI_Y_VAL_PATH', 'MODELS_PATH', 'MODEL_NAME', 'MODEL_PATH', 'MODEL_VARIANT', 'NUM_CHOICES', 'NUM_LAYERS', 'NUM_WORKERS', 'PIN_MEMORY', 'Path', 'QUALITY_TOLERANCE_ABSOLUTE', 'REPORT_JSON_PATH', 'RESULTS_PATH', 'ROUTED_MODEL_PATH', 'ROUTED_MODEL_SAVE_PATH', 'ROUTER_HIDDEN_SIZES', 'ROUTER_LEARNING_RATE', 'ROUTER_LOSS_EPS', 'ROUTER_WEIGHT_DECAY', 'ROUTING_ANALYSIS_PATH', 'ROUTING_AVAILABILITY', 'ROUTING_TRACE_PATH', 'RUN_STARTED_AT', 'RUN_TIMESTAMP', 'ReduceLROnPlateau', 'RoutedBertModel', 'RoutedFlopCounter', 'SHOW_BATCH_PROGRESS', 'SPLIT_TO_PRINT', 'SST_BACKBONE_PATH', 'SST_COMPUTE_COST_WEIGHT', 'SST_DATASET_PATH', 'SST_EXPERT_PATH', 'SST_X_TEST_PATH', 'SST_X_TRAIN_PATH', 'SST_X_VAL_PATH', 'SST_Y_TEST_PATH', 'SST_Y_TRAIN_PATH', 'SST_Y_VAL_PATH', 'SharedMLPRouter', 'TAU_ANNEALING_EPOCHS', 'TAU_END', 'TAU_EVAL', 'TAU_START', 'TOKENIZER_NAME', 'TRAINED_EXPERTS_PATH', 'TRAINING_HISTORY_PATH', 'TRAINING_SUMMARY_PATH', 'TensorDataset', 'TrainingFlopCounter', 'WEIGHT_DECAY', 'X_mnli_test', 'X_mnli_train', 'X_mnli_val', 'X_sst_test', 'X_sst_train', 'X_sst_val', 'accuracy_score', 'add_inference_cost', 'add_training_cost', 'all_correct', 'all_final_cos', 'all_final_l2', 'all_losses', 'all_merged_counts', 'all_switches', 'analysis', 'attention_mask', 'average_test', 'baseline_folder', 'baseline_report_paths', 'batch', 'batch_idx', 'batch_size', 'best_efficiency_checkpoint', 'best_epoch', 'best_quality_checkpoint', 'best_routing_analysis', 'best_routing_stats', 'best_val_f1_avg', 'best_validation_metrics', 'ce_none', 'choice_idx', 'choices', 'choices_per_layer', 'classification_criterion', 'cls_final', 'cls_hidden', 'cls_mnli', 'cls_routed', 'comparison_manifest', 'compatibility_analysis', 'compute_metrics', 'compute_total_loss', 'confusion_matrix', 'correct', 'cos', 'count_total_params', 'count_trainable_params', 'criterion', 'current_lrs', 'current_training_cost', 'datetime', 'deepcopy', 'defaultdict', 'depth', 'early_stopping', 'efficiency', 'efficiency_candidate', 'epoch', 'epoch_cost_by_task', 'epoch_record', 'evaluate', 'evaluate_original_expert_reference', 'evaluate_task', 'evaluation_results', 'experiment_config', 'extended_attention_mask', 'extract_hidden', 'extract_layer_hidden_states', 'f', 'file_sha256', 'final_cos', 'final_l2', 'finalize_routing_counter', 'finish_inference_cost', 'finish_multitask_training_cost', 'finish_routed_training_cost', 'finish_training_cost', 'finite_correlation', 'format_flops', 'freeze_module', 'gc', 'get_optimizer_lrs', 'get_tau', 'global_decision_counts', 'global_example_counts', 'group', 'group_name', 'group_params', 'hard_choice', 'hashlib', 'hidden_mnli_only', 'hidden_routed', 'i', 'init_routing_counter', 'input_ids', 'is_best', 'is_best_efficiency', 'isfinite', 'json', 'json_config_value', 'key', 'l2', 'labels', 'layer_cos_sums', 'layer_idx', 'layer_l2_sums', 'load_json', 'loader', 'loaders', 'logits', 'loss_weights', 'losses', 'mask', 'math', 'merge_cost', 'merge_float_elements', 'merge_float_tensors', 'merge_routing_counts', 'merge_training_cost', 'merged_backbone', 'merged_choices_for_layer', 'merged_config', 'merged_counts', 'merged_layers', 'metrics_by_task', 'mixed_batch_iterator', 'mnli_backbone', 'mnli_head', 'mnli_head_params', 'mnli_layers', 'mnli_param', 'mnli_results', 'mnli_routing_stats', 'mnli_state', 'mnli_test_dataset', 'mnli_test_loader', 'mnli_train_dataset', 'mnli_train_loader', 'mnli_val_dataset', 'mnli_val_loader', 'mnli_val_metrics', 'module', 'move_batch_to_device', 'name', 'new_inference_cost', 'new_training_cost', 'nn', 'np', 'old_efficiency', 'optimizer', 'param', 'path', 'pd', 'precision_recall_fscore_support', 'preds', 'print_clean_report', 'print_computational_cost', 'print_confusion_matrix', 'print_epoch_compact', 'print_module_status', 'print_performance_analysis', 'print_routing_analysis', 'print_routing_analysis_for_task', 'print_section', 'print_subsection', 'print_task_performance', 'prod', 'q1', 'q2', 'random', 'raw', 'record', 'reference', 'report', 'result', 'routed_model', 'router', 'router_logits', 'router_params', 'routing_epoch_record', 'routing_history', 'routing_trace_row', 'sample_batch', 'sample_task', 'save_best_model', 'save_training_summary', 'schedule_endpoint', 'scheduled_weight', 'scheduler', 'selected_checkpoint', 'selected_kind', 'selection_cost', 'set_routed_model_eval_mode', 'set_routed_model_train_mode', 'shared_embeddings', 'snapshot', 'soft_probs', 'split', 'sst_backbone', 'sst_head', 'sst_head_params', 'sst_layers', 'sst_param', 'sst_results', 'sst_routing_stats', 'sst_state', 'sst_test_dataset', 'sst_test_loader', 'sst_train_dataset', 'sst_train_loader', 'sst_val_dataset', 'sst_val_loader', 'sst_val_metrics', 'start_time', 'summarize_group', 'switches', 't', 'take_debug_subset', 'target', 'task', 'tau', 'texts', 'time', 'token_type_ids', 'tokenize_texts', 'tokenizer', 'tokenizer_snapshot', 'torch', 'total_examples', 'total_optimizer_params', 'tqdm', 'train_metrics', 'train_one_epoch', 'train_test_split', 'trainable_params_model', 'training_cost_by_task', 'training_history', 'training_summary', 'transformers_version', 'tree_leaves', 'unfreeze_module', 'update_routing_counter', 'val_acc_avg', 'val_f1_avg', 'val_loss_avg', 'validate_merged_layout', 'validation_references', 'value', 'x_mnli_test', 'x_mnli_test_run', 'x_mnli_train', 'x_mnli_train_run', 'x_mnli_val', 'x_mnli_val_run', 'x_sst_test', 'x_sst_test_run', 'x_sst_train', 'x_sst_train_run', 'x_sst_val', 'x_sst_val_run', 'y_mnli_test', 'y_mnli_test_run', 'y_mnli_train', 'y_mnli_train_run', 'y_mnli_val', 'y_mnli_val_run', 'y_sst_test', 'y_sst_test_run', 'y_sst_train', 'y_sst_train_run', 'y_sst_val', 'y_sst_val_run']

def run_three_seeds():
    import gc
    import json
    import random
    import numpy as np
    import torch
    from pathlib import Path
    global SEED, SEED_INDEX, EXPERIMENT_TIMESTAMP, EXECUTION_STATE, EXECUTION_MANIFEST_PATH
    if len(RUN_SEEDS) != 3 or len(set(RUN_SEEDS)) != 3:
        raise ValueError("Configure exactly three distinct training seeds")
    root = Path(ROUTED_BASELINES_DIRECTORY) / "RoutedModel"
    with execution_lock(root):
        EXECUTION_MANIFEST_PATH, EXECUTION_STATE = choose_execution(root, RUN_SEEDS)
        EXPERIMENT_TIMESTAMP = EXECUTION_STATE["timestamp"]
        experiment_results = root / "Results" / EXPERIMENT_TIMESTAMP
        for seed_index, seed_value in enumerate(RUN_SEEDS, 1):
            SEED, SEED_INDEX = seed_value, seed_index
            if EXECUTION_STATE["seed_states"][str(seed_index)]["status"] == "completed":
                folder = experiment_results / f"{EXPERIMENT_TIMESTAMP}_seed_{seed_index}"
                manifest = json.loads((folder / "comparison_manifest.json").read_text(encoding="utf-8"))
                if (manifest.get("measurement_status") != "completed"
                        or manifest.get("experiment_group", {}).get("seed") != seed_value
                        or not (folder / Path(manifest["report_path"]).name).is_file()):
                    raise ValueError(f"Risultati mancanti o incompleti per il seed {seed_value}: {folder}")
                print(f"Seed {seed_value} gia' completato: salto training e valutazione.")
                continue
            random.seed(SEED)
            np.random.seed(SEED)
            torch.manual_seed(SEED)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(SEED)
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True
            print(f"\n===== ROUTED SEED {SEED_INDEX}/3: {SEED} =====")
            record_execution_seed("initializing")
            try:
                initialize_paths()
                load_datasets()
                select_dataset_subset()
                load_models_and_heads()
                extract_layers()
                initialize_router()
                initialize_routed_model()
                initialize_optimizer_and_counters()
                prepare_dataloaders()
                configure_loss()
                train_routed()
                record_execution_seed("evaluating")
                load_selected_checkpoint()
                evaluate_selected_checkpoint()
                measure_routed_and_fixed_experts()
                print_run_report()
                check_mnli_representations()
                record_execution_seed("completed", len(training_history))
            finally:
                # Release seed objects also on a recoverable exception or Ctrl+C.
                for state_name in RUN_STATE_NAMES:
                    globals().pop(state_name, None)
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        save_three_seed_summary(experiment_results, EXPERIMENT_TIMESTAMP, RUN_SEEDS)
        EXECUTION_STATE["status"] = "completed"
        atomic_save_json(EXECUTION_MANIFEST_PATH, EXECUTION_STATE)
        print("Esecuzione completata:", EXPERIMENT_TIMESTAMP)


run_three_seeds()
